In [2]:
:dep cp_sat = "0.4.1"
// Parameters
let batch_mode = true;

# App-01 : Le problème des N-Reines en Rust

**Navigation** : [Index](../README.md) | [Part2-CSP](../Part2-CSP/README.md) | [App-2 GraphColoring >>](App-02-GraphColoring-Python.ipynb) | [Version Python](App-01-NQueens-Python.ipynb) | [Version C#](App-01-NQueens-CSharp.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Formaliser** le problème des N-Reines comme un CSP (variables, domaines, contraintes) -- *Bloom : Comprendre*
2. **Implementer** un solveur par backtracking avec heuristiques (MRV, Forward Checking) -- *Bloom : Appliquer*
3. **Comparer** backtracking, min-conflicts et OR-Tools CP-SAT sur différentes tailles -- *Bloom : Analyser*
4. **Evaluer** les forces et limites de chaque approche selon la taille du problème -- *Bloom : Evaluer*

### Prerequis
- Rust, noyau Jupyter `rust` (evcxr), `cp_sat` 0.4.1, compilateur C++ et installation système d'OR-Tools et de protobuf (`ORTOOLS_PREFIX=/opt/ortools`, `RUSTFLAGS='-Clink-arg=-lprotobuf'` dans l'environnement qui lance Jupyter)
- [CSP-1 : Fundamentals](../Part2-CSP/CSP-01-Fundamentals-Python.ipynb) -- backtracking, MRV, formalisme CSP

***

## 1. Introduction

Le problème des **N-Reines** est l'un des benchmarks les plus celebres en informatique et en intelligence artificielle. L'enonce est simple :

> Placer $N$ reines sur un echiquier $N \times N$ de sorte qu'aucune paire de reines ne s'attaque mutuellement (ni même ligne, ni même colonne, ni même diagonale).

### Historique

| Date | Auteur | Contribution |
|------|--------|--------------|
| 1848 | Max Bezzel | Pose le problème pour N=8 dans un journal d'echecs |
| 1850 | Franz Nauck | Première solution complète et generalisation a N reines |
| 1874 | S. Gunther & J.W.L. Glaisher | Première approche systématique par determinants |
| 1992 | Sosic & Gu | Min-Conflicts : resolution quasi-lineaire en temps |

### Pourquoi ce problème est important

- **Benchmark CSP** : il sert a evaluer les algorithmes de satisfaction de contraintes
- **Scalabilite** : l'espace de recherche croit exponentiellement ($N^N$ naif), mais des algorithmes intelligents resolvent $N = 10^6$ en secondes
- **Pedagogie** : il illustre parfaitement la différence entre recherche systématique et recherche locale

In [3]:
// Imports pour tout le notebook
{
    // Helpers partages de la serie Search
    // sys.path.insert(0, os.path.abspath('..'))
    // from search_helpers import benchmark_table, plot_benchmark
    // Les helpers Python ci-dessus restent des repères ; les cellules Rust sont autonomes.
    use std::mem;

    struct Mt19937 {
        state: [u32; 624],
        index: usize,
    }

    impl Mt19937 {
        fn seeded(seed: u64) -> Self {
            // CPython initialise par tableau : mots de 32 bits, le moins significatif en premier.
            let key = [seed as u32, (seed >> 32) as u32];
            let key_len = if key[1] == 0 { 1 } else { 2 };
            let mut state = [0u32; 624];
            state[0] = 19_650_218;
            for i in 1..624 {
                let previous = state[i - 1];
                state[i] = 1_812_433_253u32
                    .wrapping_mul(previous ^ (previous >> 30))
                    .wrapping_add(i as u32);
            }
            let (mut i, mut j) = (1, 0);
            for _ in 0..624 {
                let previous = state[i - 1];
                state[i] = (state[i] ^ (previous ^ (previous >> 30)).wrapping_mul(1_664_525))
                    .wrapping_add(key[j])
                    .wrapping_add(j as u32);
                i += 1;
                if i == 624 {
                    state[0] = state[623];
                    i = 1;
                }
                j = (j + 1) % key_len;
            }
            for _ in 0..623 {
                let previous = state[i - 1];
                state[i] = (state[i] ^ (previous ^ (previous >> 30)).wrapping_mul(1_566_083_941))
                    .wrapping_sub(i as u32);
                i += 1;
                if i == 624 {
                    state[0] = state[623];
                    i = 1;
                }
            }
            state[0] = 0x8000_0000;
            Self { state, index: 624 }
        }

        fn next_u32(&mut self) -> u32 {
            if self.index == 624 {
                for i in 0..624 {
                    let bits = (self.state[i] & 0x8000_0000)
                        | (self.state[(i + 1) % 624] & 0x7fff_ffff);
                    let odd_mask = if bits & 1 == 0 { 0 } else { 0x9908_b0df };
                    self.state[i] = self.state[(i + 397) % 624] ^ (bits >> 1) ^ odd_mask;
                }
                self.index = 0;
            }
            let mut bits = self.state[self.index];
            self.index += 1;
            bits ^= bits >> 11;
            bits ^= (bits << 7) & 0x9d2c_5680;
            bits ^= (bits << 15) & 0xefc6_0000;
            bits ^ (bits >> 18)
        }

        fn getrandbits(&mut self, k: u32) -> u32 {
            assert!(k <= u32::BITS);
            if k == 0 { 0 } else { self.next_u32() >> (u32::BITS - k) }
        }

        fn randbelow(&mut self, n: usize) -> usize {
            assert!(n > 0 && n <= u32::MAX as usize);
            let n = n as u32;
            // Même pour une puissance de deux, CPython demande n.bit_length() bits.
            let k = u32::BITS - n.leading_zeros();
            loop {
                let value = self.getrandbits(k);
                if value < n {
                    return value as usize;
                }
            }
        }

        fn shuffle<T>(&mut self, values: &mut [T]) {
            for i in (1..values.len()).rev() {
                values.swap(i, self.randbelow(i + 1));
            }
        }

        fn choice<'a, T>(&mut self, values: &'a [T]) -> &'a T {
            &values[self.randbelow(values.len())]
        }
    }

    // Seed pour reproductibilite
    let mut rng = Mt19937::seeded(42);
    let mut rows = [0, 1, 2, 3, 4, 5, 6, 7];
    rng.shuffle(&mut rows);
    assert_eq!(rows, [3, 4, 6, 7, 2, 5, 0, 1]);
    assert_eq!(*rng.choice(&rows[..1]), rows[0]);
    println!("Imports OK");
};


Imports OK


***

## 2. Formulation comme CSP

### Modelisation

La cle d'une resolution efficace est de choisir une bonne representation. Puisque chaque colonne doit contenir exactement une reine, on utilise :

| Composant CSP | Définition | Taille |
|---------------|------------|--------|
| **Variables** | $Q_0, Q_1, \ldots, Q_{N-1}$ (une par colonne) | $N$ |
| **Domaines** | $D_i = \{0, 1, \ldots, N-1\}$ (numéro de ligne) | $N$ valeurs chacun |
| **Contraintes** | Pour tout $i \neq j$ : $Q_i \neq Q_j$ et $\vert i - j\vert \neq \vert Q_i - Q_j\vert$ | $\binom{N}{2}$ paires |

### Espace de recherche

| Representation | Taille | Explication |
|----------------|--------|-------------|
| Naive (case libre) | $\binom{N^2}{N}$ | Combinaisons de N cases parmi $N^2$ |
| Une reine par colonne | $N^N$ | Chaque colonne choisit une ligne |
| Permutation | $N!$ | Contrainte AllDifferent sur les lignes |

Pour $N = 8$ : $N^N = 16\,777\,216$ tandis que $N! = 40\,320$ -- un facteur de reduction de 416x rien qu'avec la modelisation.

### Contraintes detaillees

Pour deux reines aux colonnes $i$ et $j$ (avec $i < j$), il faut :

$$Q_i \neq Q_j \quad \text{(pas même ligne)}$$
$$|Q_i - Q_j| \neq |i - j| \quad \text{(pas même diagonale)}$$

La contrainte de colonne est déjà satisfaite par construction (une variable par colonne).

Implementons d'abord une fonction de visualisation de l'echiquier, puis verifions visuellement une solution connue pour $N = 8$.

In [4]:
// Dessiner un échiquier en texte
{
    fn draw_queens_text(queens: &[usize], n: usize, title: &str) {
        assert_eq!(queens.len(), n);
        assert!(queens.iter().all(|&row| row < n));
        println!("{title}");

        // Dessiner l'echiquier
        let mut board = vec![vec!['.'; n]; n];
        for (row, cells) in board.iter_mut().enumerate() {
            for (col, cell) in cells.iter_mut().enumerate() {
                if (row + col) % 2 != 0 {
                    *cell = '#';
                }
            }
        }

        // Placer les reines
        // queens[col] est le numéro de ligne de la reine dans cette colonne.
        for (col, &row) in queens.iter().enumerate() {
            board[row][col] = '♛';
        }
        for cells in board {
            println!("{}", cells.into_iter().collect::<String>());
        }

        // Lignes d'attaque (pour petits echiquiers)
        // Dans la version texte, on indique les paires en conflit sous l'échiquier.
        if n <= 12 {
            let mut conflicts = 0;
            for (left, &row_left) in queens.iter().enumerate() {
                for (right, &row_right) in queens.iter().enumerate().skip(left + 1) {
                    if row_left == row_right || row_left.abs_diff(row_right) == right - left {
                        println!("Attaque : ({left}, {row_left}) ↔ ({right}, {row_right})");
                        conflicts += 1;
                    }
                }
            }
            println!("Paires en conflit : {conflicts}");
        }
    }

    // Afficher une solution connue pour N=8
    let known_solution_8 = [0, 4, 7, 5, 2, 6, 1, 3];
    draw_queens_text(&known_solution_8, 8, "Solution connue des 8-Reines");
};


Solution connue des 8-Reines


♛#.#.#.#


#.#.#.♛.


.#.#♛#.#


#.#.#.#♛


.♛.#.#.#


#.#♛#.#.


.#.#.♛.#


#.♛.#.#.


Paires en conflit : 0


### Interprétation : visualisation de la solution

L’échiquier textuel affiche huit reines (`♛`) et indique `Paires en conflit : 0`. Le placement `[0, 4, 7, 5, 2, 6, 1, 3]` est donc une solution valide.

| Vérification | Résultat |
|-------------|----------|
| Lignes distinctes | Oui (0, 1, 2, 3, 4, 5, 6, 7 : une permutation) |
| Colonnes distinctes | Oui (par construction : une reine par colonne) |
| Diagonales | Aucune paire en attaque diagonale |

> **Remarque** : pour $N = 8$, il existe exactement **92 solutions distinctes** (12 fondamentales à symétrie près). Nous les énumérerons dans la section CP-SAT.

***

## 3. Approche 1 : Backtracking

Le backtracking est la méthode de reference pour les CSP. Nous allons implementer trois variantes de complexite croissante :

1. **Backtracking simple** : exploration brute, colonne par colonne
2. **Avec heuristique MRV** : choisir la colonne au domaine le plus restreint
3. **Avec Forward Checking** : propager les contraintes pour reduire les domaines

### 3.1 Backtracking simple

L'algorithme place une reine dans chaque colonne de gauche a droite. Pour chaque ligne candidate, il verifie si la reine est en conflit avec les reines déjà placees.

In [5]:
// Backtracking simple : une colonne à la fois
{
    fn is_safe(queens: &[usize], col: usize, row: usize) -> bool {
        for (c, &r) in queens[..col].iter().enumerate() {
            if r == row { // meme ligne
                return false;
            }
            if c.abs_diff(col) == r.abs_diff(row) { // meme diagonale
                return false;
            }
        }
        true
    }

    fn backtracking_simple(n: usize) -> Option<(Vec<usize>, usize)> {
        fn solve(queens: &mut [usize], col: usize, nodes: &mut usize) -> bool {
            if col == queens.len() {
                return true;
            }
            for row in 0..queens.len() {
                *nodes += 1;
                if is_safe(queens, col, row) {
                    queens[col] = row;
                    if solve(queens, col + 1, nodes) {
                        return true;
                    }
                }
            }
            false
        }

        let mut queens = vec![0; n];
        let mut nodes = 0; // compteur mutable
        solve(&mut queens, 0, &mut nodes).then_some((queens, nodes))
    }

    // Test rapide sur N=8
    let (solution, nodes) = backtracking_simple(8).expect("Solution attendue pour N=8");
    assert_eq!(solution.as_slice(), &[0, 4, 7, 5, 2, 6, 1, 3]);
    assert_eq!(nodes, 876);
    println!("Solution 8-Reines : {solution:?}");
    println!("Noeuds explores   : {nodes}");
};


Solution 8-Reines : [0, 4, 7, 5, 2, 6, 1, 3]


Noeuds explores   : 876


### 3.2 Avec heuristique MRV

L'heuristique **MRV** (Minimum Remaining Values) choisit a chaque étape la variable (colonne) dont le domaine restant est le plus petit. C'est la stratégie **fail-first** : on detecte les impasses le plus tot possible.

Pour N-Reines, cela signifie choisir la colonne qui a le moins de lignes viables.

In [6]:
// Backtracking avec MRV
{
    fn backtracking_mrv(n: usize) -> Option<(Vec<usize>, usize)> {
        fn get_viable(queens: &[Option<usize>], col: usize) -> Vec<usize> {
            (0..queens.len())
                .filter(|&row| {
                    queens.iter().enumerate().all(|(c, &r)| match r {
                        None => true,
                        Some(r) => r != row && c.abs_diff(col) != r.abs_diff(row),
                    })
                })
                .collect()
        }

        fn solve(queens: &mut [Option<usize>], nodes: &mut usize) -> bool {
            // MRV : choisir la colonne avec le moins de valeurs viables
            // En cas d'égalité, prendre la colonne de plus petit indice.
            let choice = (0..queens.len())
                .filter(|&col| queens[col].is_none())
                .map(|col| (col, get_viable(queens, col)))
                .min_by_key(|(col, rows)| (rows.len(), *col));
            let Some((col, rows)) = choice else { return true; };

            for row in rows {
                *nodes += 1;
                queens[col] = Some(row);
                if solve(queens, nodes) {
                    return true;
                }
                queens[col] = None;
            }
            false
        }

        let mut queens = vec![None; n];
        let mut nodes = 0;
        if solve(&mut queens, &mut nodes) {
            queens.into_iter().collect::<Option<Vec<_>>>().map(|rows| (rows, nodes))
        } else {
            None
        }
    }

    let (solution, nodes) = backtracking_mrv(8).expect("Solution attendue pour N=8");
    println!("Solution (MRV)    : {solution:?}");
    println!("Noeuds explores   : {nodes}");
};


Solution (MRV)    : [0, 4, 7, 5, 2, 6, 1, 3]


Noeuds explores   : 75


### 3.3 Avec Forward Checking

Le **Forward Checking** va plus loin que MRV : a chaque assignation, il propage immediatement les contraintes pour retirer des domaines des variables non assignees toute valeur rendue impossible. Si un domaine devient vide, on backtrack immediatement sans explorer le sous-arbre.

| Technique | Quand detecte-t-on l'echec ? |
|-----------|------------------------------|
| Backtracking simple | Au moment ou on essaie une valeur |
| Forward Checking | Des qu'un domaine voisin devient vide |

In [7]:
// Backtracking avec Forward Checking et MRV
{
    // Pour comparer les nœuds, on reproduit l'ordre des ensembles CPython
    // de petits entiers : parcours par case, suppressions et copie compacte.
    #[derive(Clone, Copy, PartialEq, Eq)]
    enum Slot { Empty, Deleted, Row(usize) }

    #[derive(Clone)]
    struct Domain { slots: Vec<Slot>, len: usize, fill: usize }

    impl Domain {
        fn empty() -> Self {
            Self { slots: vec![Slot::Empty; 8], len: 0, fill: 0 }
        }

        fn rows(&self) -> impl Iterator<Item = usize> + '_ {
            self.slots.iter().filter_map(|slot| match *slot {
                Slot::Row(row) => Some(row),
                _ => None,
            })
        }

        fn locate(&self, row: usize) -> Result<usize, usize> {
            // Le hachage d'un petit entier positif est l'entier lui-même.
            let mask = self.slots.len() - 1;
            let mut perturb = row;
            let mut i = row & mask;
            let mut deleted = None;
            loop {
                let end = if i + 9 <= mask { i + 9 } else { i };
                for j in i..=end {
                    match self.slots[j] {
                        Slot::Row(value) if value == row => return Ok(j),
                        Slot::Empty => return Err(deleted.unwrap_or(j)),
                        Slot::Deleted => deleted = Some(j),
                        _ => {}
                    }
                }
                perturb >>= 5;
                i = i.wrapping_mul(5).wrapping_add(1).wrapping_add(perturb) & mask;
            }
        }

        fn insert_without_resize(&mut self, row: usize) {
            if let Err(i) = self.locate(row) {
                if self.slots[i] == Slot::Empty { self.fill += 1; }
                self.slots[i] = Slot::Row(row);
                self.len += 1;
            }
        }

        fn rehash(&self, min_used: usize) -> Self {
            let mut size = 8;
            while size <= min_used { size *= 2; }
            let mut out = Self { slots: vec![Slot::Empty; size], len: 0, fill: 0 };
            for row in self.rows() { out.insert_without_resize(row); }
            out
        }

        fn insert(&mut self, row: usize) {
            let old_fill = self.fill;
            self.insert_without_resize(row);
            if self.fill != old_fill && self.fill * 5 >= (self.slots.len() - 1) * 3 {
                *self = self.rehash(self.len * 4);
            }
        }

        fn remove(&mut self, row: usize) -> bool {
            if let Ok(i) = self.locate(row) {
                self.slots[i] = Slot::Deleted;
                self.len -= 1;
                true
            } else {
                false
            }
        }

        fn copy_domain(&self) -> Self {
            let min_used = if self.len * 5 >= 7 * 3 { self.len * 2 } else { 0 };
            let mut size = 8;
            while size <= min_used { size *= 2; }
            if size == self.slots.len() && self.fill == self.len {
                self.clone()
            } else {
                self.rehash(min_used)
            }
        }
    }

    fn backtracking_fc(n: usize) -> Option<(Vec<usize>, usize)> {
        // domains[col] = ensemble des lignes encore possibles
        let mut full = Domain::empty();
        for row in 0..n { full.insert(row); }
        let mut domains = vec![full; n];
        let mut queens = vec![None; n];
        let mut nodes = 0;

        fn forward_check(queens: &[Option<usize>], domains: &mut [Domain], col: usize, row: usize) -> Vec<(usize, usize)> {
            let mut pruned = Vec::new();
            for c in 0..queens.len() {
                if queens[c].is_some() { continue; }
                // Retirer la meme ligne
                if domains[c].remove(row) { pruned.push((c, row)); }
                // Retirer les diagonales
                let d = c.abs_diff(col);
                for r in [row.checked_sub(d), row.checked_add(d)].into_iter().flatten() {
                    if r < queens.len() && domains[c].remove(r) { pruned.push((c, r)); }
                }
            }
            pruned
        }

        fn solve(queens: &mut [Option<usize>], domains: &mut [Domain], nodes: &mut usize) -> bool {
            // MRV : choisir la colonne non assignee au domaine le plus petit
            let Some(col) = (0..queens.len())
                .filter(|&c| queens[c].is_none())
                .min_by_key(|&c| (domains[c].len, c)) else { return true; };
            let rows: Vec<_> = domains[col].rows().collect();
            for row in rows {
                *nodes += 1;
                queens[col] = Some(row); // assignee
                let old_domain = domains[col].copy_domain();
                domains[col] = Domain::empty();
                let pruned = forward_check(queens, domains, col, row);
                // Verifier qu'aucun domaine futur n'est vide
                let empty = (0..queens.len()).any(|c| queens[c].is_none() && domains[c].len == 0);
                if !empty && solve(queens, domains, nodes) { return true; }
                // Restauration
                for (c, r) in pruned { domains[c].insert(r); }
                domains[col] = old_domain;
                queens[col] = None;
            }
            false
        }

        if solve(&mut queens, &mut domains, &mut nodes) {
            queens.into_iter().collect::<Option<Vec<_>>>().map(|rows| (rows, nodes))
        } else {
            None
        }
    }

    let (solution, nodes) = backtracking_fc(8).expect("Solution attendue pour N=8");
    println!("Solution (FC+MRV) : {solution:?}");
    println!("Noeuds explores   : {nodes}");
};


Solution (FC+MRV) : [0, 4, 7, 5, 2, 6, 1, 3]


Noeuds explores   : 75


### 3.4 Benchmark des variantes de backtracking

Comparons les trois variantes sur différentes tailles de problème. Le nombre de noeuds explores et le temps d'exécution mesurent l'efficacite.

In [8]:
// Benchmark des trois variantes de backtracking
{
    use std::time::Instant;

    type SearchResult = Option<(Vec<usize>, usize)>;
    type Solver = fn(usize) -> SearchResult;

    // Le solveur simple compte chaque ligne essayée, même si elle est interdite.
    fn backtracking_simple(n: usize) -> SearchResult {
        fn solve(q: &mut [usize], col: usize, nodes: &mut usize) -> bool {
            if col == q.len() { return true; }
            for row in 0..q.len() {
                *nodes += 1;
                if q[..col].iter().enumerate().all(|(c, &r)| r != row && col - c != r.abs_diff(row)) {
                    q[col] = row;
                    if solve(q, col + 1, nodes) { return true; }
                }
            }
            false
        }
        let mut q = vec![0; n];
        let mut nodes = 0;
        solve(&mut q, 0, &mut nodes).then_some((q, nodes))
    }

    // MRV ne compte que les candidats viables qu'il essaie d'affecter.
    fn backtracking_mrv(n: usize) -> SearchResult {
        fn solve(q: &mut [Option<usize>], nodes: &mut usize) -> bool {
            let choice = (0..q.len()).filter(|&c| q[c].is_none()).map(|c| {
                let rows: Vec<_> = (0..q.len()).filter(|&r| {
                    q.iter().enumerate().all(|(other, &assigned)| match assigned {
                        None => true,
                        Some(y) => y != r && other.abs_diff(c) != y.abs_diff(r),
                    })
                }).collect();
                (c, rows)
            }).min_by_key(|(c, rows)| (rows.len(), *c));
            let Some((col, rows)) = choice else { return true; };
            for row in rows {
                *nodes += 1;
                q[col] = Some(row);
                if solve(q, nodes) { return true; }
                q[col] = None;
            }
            false
        }
        let mut q = vec![None; n];
        let mut nodes = 0;
        if solve(&mut q, &mut nodes) {
            q.into_iter().collect::<Option<Vec<_>>>().map(|rows| (rows, nodes))
        } else {
            None
        }
    }

    // FC+MRV reprend l'ordre des ensembles CPython de petits entiers.
    // Un ordre croissant peut changer la première solution et le compte des nœuds.
    #[derive(Clone, Copy, PartialEq, Eq)]
    enum Slot { Empty, Deleted, Row(usize) }

    #[derive(Clone)]
    struct Domain { slots: Vec<Slot>, len: usize, fill: usize }

    impl Domain {
        fn empty() -> Self {
            Self { slots: vec![Slot::Empty; 8], len: 0, fill: 0 }
        }

        fn rows(&self) -> impl Iterator<Item = usize> + '_ {
            self.slots.iter().filter_map(|slot| match *slot {
                Slot::Row(row) => Some(row),
                _ => None,
            })
        }

        fn locate(&self, row: usize) -> Result<usize, usize> {
            let mask = self.slots.len() - 1;
            let mut perturb = row;
            let mut i = row & mask;
            let mut deleted = None;
            loop {
                let end = if i + 9 <= mask { i + 9 } else { i };
                for j in i..=end {
                    match self.slots[j] {
                        Slot::Row(value) if value == row => return Ok(j),
                        Slot::Empty => return Err(deleted.unwrap_or(j)),
                        Slot::Deleted => deleted = Some(j),
                        _ => {}
                    }
                }
                perturb >>= 5;
                i = i.wrapping_mul(5).wrapping_add(1).wrapping_add(perturb) & mask;
            }
        }

        fn insert_without_resize(&mut self, row: usize) {
            if let Err(i) = self.locate(row) {
                if self.slots[i] == Slot::Empty { self.fill += 1; }
                self.slots[i] = Slot::Row(row);
                self.len += 1;
            }
        }

        fn rehash(&self, min_used: usize) -> Self {
            let mut size = 8;
            while size <= min_used { size *= 2; }
            let mut out = Self { slots: vec![Slot::Empty; size], len: 0, fill: 0 };
            for row in self.rows() { out.insert_without_resize(row); }
            out
        }

        fn insert(&mut self, row: usize) {
            let old_fill = self.fill;
            self.insert_without_resize(row);
            if self.fill != old_fill && self.fill * 5 >= (self.slots.len() - 1) * 3 {
                *self = self.rehash(self.len * 4);
            }
        }

        fn remove(&mut self, row: usize) -> bool {
            if let Ok(i) = self.locate(row) {
                self.slots[i] = Slot::Deleted;
                self.len -= 1;
                true
            } else {
                false
            }
        }

        fn copy_domain(&self) -> Self {
            let min_used = if self.len * 5 >= 7 * 3 { self.len * 2 } else { 0 };
            let mut size = 8;
            while size <= min_used { size *= 2; }
            if size == self.slots.len() && self.fill == self.len {
                self.clone()
            } else {
                self.rehash(min_used)
            }
        }
    }

    fn backtracking_fc(n: usize) -> SearchResult {
        fn solve(q: &mut [Option<usize>], domains: &mut [Domain], nodes: &mut usize) -> bool {
            let Some(col) = (0..q.len()).filter(|&c| q[c].is_none())
                .min_by_key(|&c| (domains[c].len, c)) else { return true; };
            let rows: Vec<_> = domains[col].rows().collect();
            for row in rows {
                *nodes += 1;
                q[col] = Some(row);
                let old = domains[col].copy_domain();
                domains[col] = Domain::empty();
                let mut pruned = Vec::new();
                // Retirer les lignes et diagonales interdites, dans l'ordre du jumeau.
                for c in 0..q.len() {
                    if q[c].is_some() { continue; }
                    let d = col.abs_diff(c);
                    for r in [Some(row), row.checked_sub(d), row.checked_add(d)].into_iter().flatten() {
                        if r < q.len() && domains[c].remove(r) { pruned.push((c, r)); }
                    }
                }
                let empty = (0..q.len()).any(|c| q[c].is_none() && domains[c].len == 0);
                if !empty && solve(q, domains, nodes) { return true; }
                for (c, r) in pruned { domains[c].insert(r); }
                domains[col] = old;
                q[col] = None;
            }
            false
        }

        let mut full = Domain::empty();
        for row in 0..n { full.insert(row); }
        let mut domains = vec![full; n];
        let mut q = vec![None; n];
        let mut nodes = 0;
        if solve(&mut q, &mut domains, &mut nodes) {
            q.into_iter().collect::<Option<Vec<_>>>().map(|rows| (rows, nodes))
        } else {
            None
        }
    }

    let solvers: [Solver; 3] = [backtracking_simple, backtracking_mrv, backtracking_fc];
    let cases = [(4, [26, 8, 8]), (8, [876, 75, 75]), (12, [3066, 153, 153]), (16, [160712, 44, 39])];
    println!("{:>4}  {:>12}  {:>12}  {:>12}   (noeuds explores)", "N", "Simple", "MRV", "FC+MRV");
    let mut times = Vec::new();
    for (n, expected) in cases {
        let mut counts = [0; 3];
        let mut elapsed_ms = [0.0; 3];
        for (i, solver) in solvers.iter().enumerate() {
            let start = Instant::now();
            let result = solver(n);
            elapsed_ms[i] = start.elapsed().as_secs_f64() * 1000.0;
            counts[i] = result.expect("Solution attendue pour cette taille").1;
        }
        let [simple, mrv, fc] = counts;
        println!("{n:>4}  {simple:>12}  {mrv:>12}  {fc:>12}");
        assert_eq!(counts, expected, "Décompte pour N={n}");
        times.push((n, elapsed_ms));
    }

    println!("\n{:>4}  {:>12}  {:>12}  {:>12}   (ms)", "N", "Simple", "MRV", "FC+MRV");
    for (n, [simple, mrv, fc]) in times {
        println!("{n:>4}  {simple:>12.4}  {mrv:>12.4}  {fc:>12.4}");
    }
};


   N        Simple           MRV        FC+MRV   (noeuds explores)


   4            26             8             8


   8           876            75            75


  12          3066           153           153


  16        160712            44            39


   N        Simple           MRV        FC+MRV   (ms)


   4        0.0016        0.0068        0.0127


   8        0.0262        0.1256        0.1011


  12        0.0877        0.3829        0.3115


  16        2.2140        0.1454        0.0720


Comparons les résultats dans deux tableaux successifs — nœuds explorés, puis temps d'exécution — pour mieux apprecier les différences entre variantes.

In [9]:
// Visualisation comparative
{
    use std::time::Instant;
    type SearchResult = Option<(Vec<usize>, usize)>;
    type Solver = fn(usize) -> SearchResult;

    // Le backtracking simple compte toutes les lignes essayées.
    fn backtracking_simple(n: usize) -> SearchResult {
        fn solve(q: &mut [usize], col: usize, nodes: &mut usize) -> bool {
            if col == q.len() { return true; }
            for row in 0..q.len() {
                *nodes += 1;
                if q[..col].iter().enumerate().all(|(c, &r)| r != row && col - c != r.abs_diff(row)) {
                    q[col] = row;
                    if solve(q, col + 1, nodes) { return true; }
                }
            }
            false
        }
        let mut q = vec![0; n];
        let mut nodes = 0;
        solve(&mut q, 0, &mut nodes).then_some((q, nodes))
    }

    // MRV recalcule les lignes viables et départage les colonnes par indice.
    fn backtracking_mrv(n: usize) -> SearchResult {
        fn solve(q: &mut [Option<usize>], nodes: &mut usize) -> bool {
            let choice = (0..q.len()).filter(|&c| q[c].is_none()).map(|c| {
                let rows: Vec<_> = (0..q.len()).filter(|&r| {
                    q.iter().enumerate().all(|(other, &assigned)| match assigned {
                        None => true,
                        Some(y) => y != r && other.abs_diff(c) != y.abs_diff(r),
                    })
                }).collect();
                (c, rows)
            }).min_by_key(|(c, rows)| (rows.len(), *c));
            let Some((col, rows)) = choice else { return true; };
            for row in rows {
                *nodes += 1;
                q[col] = Some(row);
                if solve(q, nodes) { return true; }
                q[col] = None;
            }
            false
        }
        let mut q = vec![None; n];
        let mut nodes = 0;
        if solve(&mut q, &mut nodes) {
            q.into_iter().collect::<Option<Vec<_>>>().map(|rows| (rows, nodes))
        } else {
            None
        }
    }

    // FC+MRV reproduit le parcours des ensembles CPython de petits entiers,
    // y compris les cases supprimées et l'effet d'une copie sur l'ordre de visite.
    #[derive(Clone, Copy, PartialEq, Eq)]
    enum Slot { Empty, Deleted, Row(usize) }

    #[derive(Clone)]
    struct Domain { slots: Vec<Slot>, len: usize, fill: usize }

    impl Domain {
        fn empty() -> Self {
            Self { slots: vec![Slot::Empty; 8], len: 0, fill: 0 }
        }

        fn rows(&self) -> impl Iterator<Item = usize> + '_ {
            self.slots.iter().filter_map(|slot| match *slot {
                Slot::Row(row) => Some(row),
                _ => None,
            })
        }

        fn locate(&self, row: usize) -> Result<usize, usize> {
            let mask = self.slots.len() - 1;
            let mut perturb = row;
            let mut i = row & mask;
            let mut deleted = None;
            loop {
                let end = if i + 9 <= mask { i + 9 } else { i };
                for j in i..=end {
                    match self.slots[j] {
                        Slot::Row(value) if value == row => return Ok(j),
                        Slot::Empty => return Err(deleted.unwrap_or(j)),
                        Slot::Deleted => deleted = Some(j),
                        _ => {}
                    }
                }
                perturb >>= 5;
                i = i.wrapping_mul(5).wrapping_add(1).wrapping_add(perturb) & mask;
            }
        }

        fn insert_without_resize(&mut self, row: usize) {
            if let Err(i) = self.locate(row) {
                if self.slots[i] == Slot::Empty { self.fill += 1; }
                self.slots[i] = Slot::Row(row);
                self.len += 1;
            }
        }

        fn rehash(&self, min_used: usize) -> Self {
            let mut size = 8;
            while size <= min_used { size *= 2; }
            let mut out = Self { slots: vec![Slot::Empty; size], len: 0, fill: 0 };
            for row in self.rows() { out.insert_without_resize(row); }
            out
        }

        fn insert(&mut self, row: usize) {
            let old_fill = self.fill;
            self.insert_without_resize(row);
            if self.fill != old_fill && self.fill * 5 >= (self.slots.len() - 1) * 3 {
                *self = self.rehash(self.len * 4);
            }
        }

        fn remove(&mut self, row: usize) -> bool {
            if let Ok(i) = self.locate(row) {
                self.slots[i] = Slot::Deleted;
                self.len -= 1;
                true
            } else {
                false
            }
        }

        fn copy_domain(&self) -> Self {
            let min_used = if self.len * 5 >= 7 * 3 { self.len * 2 } else { 0 };
            let mut size = 8;
            while size <= min_used { size *= 2; }
            if size == self.slots.len() && self.fill == self.len {
                self.clone()
            } else {
                self.rehash(min_used)
            }
        }
    }

    fn backtracking_fc(n: usize) -> SearchResult {
        fn solve(q: &mut [Option<usize>], domains: &mut [Domain], nodes: &mut usize) -> bool {
            let Some(col) = (0..q.len()).filter(|&c| q[c].is_none())
                .min_by_key(|&c| (domains[c].len, c)) else { return true; };
            let rows: Vec<_> = domains[col].rows().collect();
            for row in rows {
                *nodes += 1;
                q[col] = Some(row);
                let old = domains[col].copy_domain();
                domains[col] = Domain::empty();
                let mut pruned = Vec::new();
                // Élaguer puis remettre uniquement les valeurs retirées à ce niveau.
                for c in 0..q.len() {
                    if q[c].is_some() { continue; }
                    let d = col.abs_diff(c);
                    for r in [Some(row), row.checked_sub(d), row.checked_add(d)].into_iter().flatten() {
                        if r < q.len() && domains[c].remove(r) { pruned.push((c, r)); }
                    }
                }
                let empty = (0..q.len()).any(|c| q[c].is_none() && domains[c].len == 0);
                if !empty && solve(q, domains, nodes) { return true; }
                for (c, r) in pruned { domains[c].insert(r); }
                domains[col] = old;
                q[col] = None;
            }
            false
        }

        let mut full = Domain::empty();
        for row in 0..n { full.insert(row); }
        let mut domains = vec![full; n];
        let mut q = vec![None; n];
        let mut nodes = 0;
        if solve(&mut q, &mut domains, &mut nodes) {
            q.into_iter().collect::<Option<Vec<_>>>().map(|rows| (rows, nodes))
        } else {
            None
        }
    }

    let solvers: [Solver; 3] = [backtracking_simple, backtracking_mrv, backtracking_fc];
    let mut results = Vec::new();
    for n in [4, 8, 12, 16] {
        let mut counts = [0; 3];
        let mut times = [0.0; 3];
        for (i, solver) in solvers.iter().enumerate() {
            let start = Instant::now();
            let result = solver(n);
            times[i] = start.elapsed().as_secs_f64() * 1_000.0;
            counts[i] = result.expect("Solution attendue pour cette taille").1;
        }
        results.push((n, counts, times));
    }

    // Noeuds explores (echelle log)
    // Le tableau texte présente les valeurs brutes plutôt qu'un axe logarithmique.
    println!("Nœuds explorés");
    println!("{:>4} {:>12} {:>12} {:>12}", "N", "Simple", "MRV", "FC+MRV");
    for &(n, [simple, mrv, fc], _) in &results {
        println!("{n:>4} {simple:>12} {mrv:>12} {fc:>12}");
    }

    // Temps d'execution
    println!("\nTemps d'exécution (ms)");
    println!("{:>4} {:>12} {:>12} {:>12}", "N", "Simple", "MRV", "FC+MRV");
    for (n, _, [simple, mrv, fc]) in results {
        println!("{n:>4} {simple:>12.4} {mrv:>12.4} {fc:>12.4}");
    }
};


Nœuds explorés


   N       Simple          MRV       FC+MRV


   4           26            8            8


   8          876           75           75


  12         3066          153          153


  16       160712           44           39


... (2 outputs left out)

   4       0.0011       0.0059       0.0116


   8       0.0198       0.1471       0.0865


  12       0.1546       0.3875       0.2362


  16       4.7120       0.1848       0.0797


### Interpretation : benchmark backtracking

**Sortie obtenue** : le nombre de noeuds et le temps augmentent exponentiellement avec $N$, mais les heuristiques reduisent fortement l'exploration.

| Variante | Principe | Impact sur N=16 |
|----------|----------|------------------|
| Simple | Ordre fixe, pas de propagation | Reference (le plus lent) |
| MRV | Fail-first : variable la plus contrainte d'abord | Reduction significative |
| FC+MRV | Propagation immediate + fail-first | Reduction maximale |

**Points cles** :
1. Forward Checking detecte les impasses plus tot en vidant les domaines
2. MRV guide la recherche vers les variables critiques
3. La combinaison FC+MRV est la plus efficace pour le backtracking

> **Limite** : même avec FC+MRV, le backtracking reste exponentiel dans le pire cas. Pour de grands $N$, il faut une approche fondamentalement différente.

***

## 4. Approche 2 : Min-Conflicts

L'algorithme **Min-Conflicts** (Minton et al., 1992) est une méthode de **recherche locale** pour les CSP. Son principe est radicalement différent du backtracking :

1. **Demarrer** avec une assignation complète aleatoire (toutes les variables ont une valeur)
2. **Tant qu'il y a des conflits** :
   - Choisir aleatoirement une variable en conflit
   - Lui assigner la valeur qui **minimise** le nombre de conflits
3. **Repeter** jusqu'a trouver une solution ou atteindre un maximum d'itérations

### Pourquoi ca marche ?

Le résultat surprenant de Sosic & Gu (1990) est que pour les N-Reines, min-conflicts trouve une solution en temps **quasi-constant** par rapport a $N$ (pour des instances aleatoires). En partant d'un placement aleatoire sur un echiquier de taille $N$, on n'a besoin que de $O(N)$ reparations en moyenne.

| Aspect | Backtracking | Min-Conflicts |
|--------|-------------|---------------|
| Type | Systématique, complet | Recherche locale, incomplet |
| Garantie de solution | Oui (si elle existe) | Non (peut rester coince) |
| Complexite typique N-Reines | Exponentielle | Quasi-lineaire |
| Demarrage | Assignation vide | Assignation complète aleatoire |

In [10]:
// Min-Conflicts reproductible pour N=8
{
    struct Mt19937 { state: [u32; 624], index: usize }

    impl Mt19937 {
        fn seeded(seed: u64) -> Self {
            // CPython initialise MT19937 par tableau de mots de 32 bits.
            let key = [seed as u32, (seed >> 32) as u32];
            let key_len = if key[1] == 0 { 1 } else { 2 };
            let mut state = [0u32; 624];
            state[0] = 19_650_218;
            for i in 1..624 {
                let p = state[i - 1];
                state[i] = 1_812_433_253u32.wrapping_mul(p ^ (p >> 30)).wrapping_add(i as u32);
            }
            let (mut i, mut j) = (1, 0);
            for _ in 0..624 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_664_525))
                    .wrapping_add(key[j]).wrapping_add(j as u32);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
                j = (j + 1) % key_len;
            }
            for _ in 0..623 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_566_083_941))
                    .wrapping_sub(i as u32);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            state[0] = 0x8000_0000;
            Self { state, index: 624 }
        }

        fn next_u32(&mut self) -> u32 {
            if self.index == 624 {
                for i in 0..624 {
                    let bits = (self.state[i] & 0x8000_0000) | (self.state[(i + 1) % 624] & 0x7fff_ffff);
                    let odd_mask = if bits & 1 == 0 { 0 } else { 0x9908_b0df };
                    self.state[i] = self.state[(i + 397) % 624] ^ (bits >> 1) ^ odd_mask;
                }
                self.index = 0;
            }
            let mut bits = self.state[self.index];
            self.index += 1;
            bits ^= bits >> 11;
            bits ^= (bits << 7) & 0x9d2c_5680;
            bits ^= (bits << 15) & 0xefc6_0000;
            bits ^ (bits >> 18)
        }

        fn randbelow(&mut self, n: usize) -> usize {
            assert!(n > 0 && n <= u32::MAX as usize);
            let n = n as u32;
            // n.leading_zeros() vaut 32 - n.bit_length() : garder les bits forts et rejeter hors plage.
            loop {
                let r = self.next_u32() >> n.leading_zeros();
                if r < n { return r as usize; }
            }
        }

        fn shuffle<T>(&mut self, values: &mut [T]) {
            for i in (1..values.len()).rev() { values.swap(i, self.randbelow(i + 1)); }
        }

        fn choice<'a, T>(&mut self, values: &'a [T]) -> &'a T {
            &values[self.randbelow(values.len())]
        }
    }

    fn conflicts_at(queens: &[usize], col: usize, row: usize) -> usize {
        queens.iter().enumerate().filter(|&(c, &r)| {
            c != col && (r == row || c.abs_diff(col) == r.abs_diff(row))
        }).count()
    }

    fn count_conflicts(queens: &[usize], col: usize) -> usize {
        conflicts_at(queens, col, queens[col])
    }

    fn total_conflicts(queens: &[usize]) -> usize {
        let mut total = 0;
        for (col, &row) in queens.iter().enumerate() {
            for (c, &r) in queens.iter().enumerate().skip(col + 1) {
                if r == row || c - col == r.abs_diff(row) { total += 1; }
            }
        }
        total
    }

    fn min_conflicts(n: usize, max_steps: Option<usize>, rng: &mut Mt19937)
        -> (Option<Vec<usize>>, usize, Vec<usize>)
    {
        let max_steps = max_steps.unwrap_or_else(|| (5 * n).max(1_000));
        // Initialisation aleatoire : une reine par colonne, ligne aleatoire
        let mut queens: Vec<_> = (0..n).collect();
        rng.shuffle(&mut queens);
        let mut history = vec![total_conflicts(&queens)];

        for step in 0..max_steps {
            // Trouver les colonnes en conflit
            let conflicted: Vec<_> = (0..n).filter(|&c| count_conflicts(&queens, c) > 0).collect();
            if conflicted.is_empty() {
                return (Some(queens), step, history); // Solution trouvee
            }

            // Choisir une colonne en conflit au hasard
            let col = *rng.choice(&conflicted);
            // Trouver la ligne qui minimise les conflits
            let mut minimum = usize::MAX;
            let mut best_rows = Vec::new();
            for row in 0..n {
                let conflicts = conflicts_at(&queens, col, row);
                if conflicts < minimum {
                    minimum = conflicts;
                    best_rows.clear();
                }
                if conflicts == minimum { best_rows.push(row); }
            }
            // Assigner la meilleure ligne (aleatoire en cas d'egalite)
            queens[col] = *rng.choice(&best_rows);
            history.push(total_conflicts(&queens));
        }
        (None, max_steps, history) // Pas de solution trouvee
    }

    // Test sur N=8
    let mut rng = Mt19937::seeded(42);
    let (solution, steps, history) = min_conflicts(8, None, &mut rng);
    let solution = solution.expect("Solution attendue avec la graine 42");
    assert_eq!(steps, 15);
    assert_eq!(history.len(), steps + 1);
    assert_eq!(total_conflicts(&solution), 0);
    println!("Solution (min-conflicts, N=8) : {solution:?}");
    println!("Iterations : {steps}");
    println!("Conflits initiaux : {}, finaux : {}", history[0], history[steps]);
};


Solution (min-conflicts, N=8) : [1, 3, 5, 7, 2, 0, 6, 4]


Iterations : 15


Conflits initiaux : 6, finaux : 0


### Le "miracle" de Min-Conflicts : scalabilite

Le résultat le plus remarquable de min-conflicts est sa capacite a résoudre des instances de **très grande taille**. Testons sur $N = 8, 50, 100, 500$ et observons l'evolution du temps.

In [11]:
// Scalabilite de min-conflicts
{
    use std::time::Instant;

    struct Mt19937 { state: [u32; 624], index: usize }

    impl Mt19937 {
        fn seeded(seed: u64) -> Self {
            // CPython initialise MT19937 par tableau de mots de 32 bits.
            let key = [seed as u32, (seed >> 32) as u32];
            let key_len = if key[1] == 0 { 1 } else { 2 };
            let mut state = [0u32; 624];
            state[0] = 19_650_218;
            for i in 1..624 {
                let p = state[i - 1];
                state[i] = 1_812_433_253u32.wrapping_mul(p ^ (p >> 30)).wrapping_add(i as u32);
            }
            let (mut i, mut j) = (1, 0);
            for _ in 0..624 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_664_525))
                    .wrapping_add(key[j]).wrapping_add(j as u32);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
                j = (j + 1) % key_len;
            }
            for _ in 0..623 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_566_083_941))
                    .wrapping_sub(i as u32);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            state[0] = 0x8000_0000;
            Self { state, index: 624 }
        }

        fn next_u32(&mut self) -> u32 {
            if self.index == 624 {
                for i in 0..624 {
                    let bits = (self.state[i] & 0x8000_0000) | (self.state[(i + 1) % 624] & 0x7fff_ffff);
                    let odd_mask = if bits & 1 == 0 { 0 } else { 0x9908_b0df };
                    self.state[i] = self.state[(i + 397) % 624] ^ (bits >> 1) ^ odd_mask;
                }
                self.index = 0;
            }
            let mut bits = self.state[self.index];
            self.index += 1;
            bits ^= bits >> 11;
            bits ^= (bits << 7) & 0x9d2c_5680;
            bits ^= (bits << 15) & 0xefc6_0000;
            bits ^ (bits >> 18)
        }

        fn randbelow(&mut self, n: usize) -> usize {
            assert!(n > 0 && n <= u32::MAX as usize);
            let n = n as u32;
            // Tirer les n.bit_length() bits forts, puis rejeter les valeurs hors plage.
            loop {
                let r = self.next_u32() >> n.leading_zeros();
                if r < n { return r as usize; }
            }
        }

        fn shuffle<T>(&mut self, values: &mut [T]) {
            for i in (1..values.len()).rev() { values.swap(i, self.randbelow(i + 1)); }
        }

        fn choice<'a, T>(&mut self, values: &'a [T]) -> &'a T {
            &values[self.randbelow(values.len())]
        }
    }

    fn conflicts_at(queens: &[usize], col: usize, row: usize) -> usize {
        queens.iter().enumerate().filter(|&(c, &r)| {
            c != col && (r == row || c.abs_diff(col) == r.abs_diff(row))
        }).count()
    }

    fn count_conflicts(queens: &[usize], col: usize) -> usize {
        conflicts_at(queens, col, queens[col])
    }

    fn total_conflicts(queens: &[usize]) -> usize {
        let mut total = 0;
        for (col, &row) in queens.iter().enumerate() {
            for (c, &r) in queens.iter().enumerate().skip(col + 1) {
                if r == row || c - col == r.abs_diff(row) { total += 1; }
            }
        }
        total
    }

    fn min_conflicts(n: usize, max_steps: usize, rng: &mut Mt19937)
        -> (Option<Vec<usize>>, usize, Vec<usize>)
    {
        // Même permutation initiale, même ordre des candidats et mêmes tirages qu'en c20.
        let mut queens: Vec<_> = (0..n).collect();
        rng.shuffle(&mut queens);
        let mut history = vec![total_conflicts(&queens)];
        for step in 0..max_steps {
            let conflicted: Vec<_> = (0..n).filter(|&c| count_conflicts(&queens, c) > 0).collect();
            if conflicted.is_empty() { return (Some(queens), step, history); }
            let col = *rng.choice(&conflicted);
            let mut minimum = usize::MAX;
            let mut best_rows = Vec::new();
            for row in 0..n {
                let conflicts = conflicts_at(&queens, col, row);
                if conflicts < minimum {
                    minimum = conflicts;
                    best_rows.clear();
                }
                if conflicts == minimum { best_rows.push(row); }
            }
            queens[col] = *rng.choice(&best_rows);
            history.push(total_conflicts(&queens));
        }
        (None, max_steps, history)
    }

    println!("{:>6}  {:>12}  {:>12}  {:>8}", "N", "Temps (ms)", "Iterations", "Trouve");
    println!("{}", "-".repeat(50));
    for (n, expected_steps) in [(8, 15), (50, 152), (100, 115), (500, 346)] {
        // Réinitialiser la graine pour chaque taille avant de démarrer le chronomètre.
        let mut rng = Mt19937::seeded(42);
        let start = Instant::now();
        let (solution, steps, history) = min_conflicts(n, 10 * n, &mut rng);
        let ms = start.elapsed().as_secs_f64() * 1_000.0;
        let found = if solution.is_some() { "Oui" } else { "Non" };
        println!("{n:>6}  {ms:>12.1}  {steps:>12}  {found:>8}");
        let solution = solution.expect("Solution attendue avec la graine 42");
        assert_eq!(steps, expected_steps, "Itérations pour N={n}");
        assert_eq!(history.last(), Some(&0));
        assert_eq!(total_conflicts(&solution), 0);
    }
};


     N    Temps (ms)    Iterations    Trouve


--------------------------------------------------


     8           0.0            15       Oui


    50           2.2           152       Oui


   100           6.0           115       Oui


   500         491.9           346       Oui


Mesurons le temps et le nombre d'itérations en fonction de $N$, dans un tableau.

In [12]:
// Visualisation : temps vs N pour min-conflicts
{
    use std::time::Instant;

    struct Mt19937 { state: [u32; 624], index: usize }

    impl Mt19937 {
        fn seeded(seed: u64) -> Self {
            // Initialisation par tableau de mots de 32 bits, comme CPython.
            let key = [seed as u32, (seed >> 32) as u32];
            let key_len = if key[1] == 0 { 1 } else { 2 };
            let mut state = [0u32; 624];
            state[0] = 19_650_218;
            for i in 1..624 {
                let p = state[i - 1];
                state[i] = 1_812_433_253u32.wrapping_mul(p ^ (p >> 30)).wrapping_add(i as u32);
            }
            let (mut i, mut j) = (1, 0);
            for _ in 0..624 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_664_525))
                    .wrapping_add(key[j]).wrapping_add(j as u32);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
                j = (j + 1) % key_len;
            }
            for _ in 0..623 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_566_083_941))
                    .wrapping_sub(i as u32);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            state[0] = 0x8000_0000;
            Self { state, index: 624 }
        }

        fn next_u32(&mut self) -> u32 {
            if self.index == 624 {
                for i in 0..624 {
                    let bits = (self.state[i] & 0x8000_0000)
                        | (self.state[(i + 1) % 624] & 0x7fff_ffff);
                    let odd_mask = if bits & 1 == 0 { 0 } else { 0x9908_b0df };
                    self.state[i] = self.state[(i + 397) % 624] ^ (bits >> 1) ^ odd_mask;
                }
                self.index = 0;
            }
            let mut bits = self.state[self.index];
            self.index += 1;
            bits ^= bits >> 11;
            bits ^= (bits << 7) & 0x9d2c_5680;
            bits ^= (bits << 15) & 0xefc6_0000;
            bits ^ (bits >> 18)
        }

        fn randbelow(&mut self, n: usize) -> usize {
            assert!(n > 0 && n <= u32::MAX as usize);
            let n = n as u32;
            // Tirer n.bit_length() bits de poids fort et rejeter hors plage.
            loop {
                let r = self.next_u32() >> n.leading_zeros();
                if r < n { return r as usize; }
            }
        }

        fn shuffle<T>(&mut self, values: &mut [T]) {
            for i in (1..values.len()).rev() { values.swap(i, self.randbelow(i + 1)); }
        }

        fn choice<'a, T>(&mut self, values: &'a [T]) -> &'a T {
            &values[self.randbelow(values.len())]
        }
    }

    fn conflicts_at(queens: &[usize], col: usize, row: usize) -> usize {
        queens.iter().enumerate().filter(|&(c, &r)| {
            c != col && (r == row || c.abs_diff(col) == r.abs_diff(row))
        }).count()
    }

    fn total_conflicts(queens: &[usize]) -> usize {
        let mut total = 0;
        for (col, &row) in queens.iter().enumerate() {
            for (c, &r) in queens.iter().enumerate().skip(col + 1) {
                if r == row || c - col == r.abs_diff(row) { total += 1; }
            }
        }
        total
    }

    fn min_conflicts(n: usize, max_steps: usize, rng: &mut Mt19937)
        -> (Option<Vec<usize>>, usize, Vec<usize>)
    {
        // Garder le même ordre des candidats et le même calcul de l'historique.
        let mut queens: Vec<_> = (0..n).collect();
        rng.shuffle(&mut queens);
        let mut history = vec![total_conflicts(&queens)];
        for step in 0..max_steps {
            let conflicted: Vec<_> = (0..n)
                .filter(|&c| conflicts_at(&queens, c, queens[c]) > 0).collect();
            if conflicted.is_empty() { return (Some(queens), step, history); }
            let col = *rng.choice(&conflicted);
            let mut minimum = usize::MAX;
            let mut best_rows = Vec::new();
            for row in 0..n {
                let conflicts = conflicts_at(&queens, col, row);
                if conflicts < minimum {
                    minimum = conflicts;
                    best_rows.clear();
                }
                if conflicts == minimum { best_rows.push(row); }
            }
            queens[col] = *rng.choice(&best_rows);
            history.push(total_conflicts(&queens));
        }
        (None, max_steps, history)
    }

    // Rejouer les mesures pour que le tableau reste autonome.
    println!("{:>6}  {:>12}  {:>12}  {:>8}", "N", "Temps (ms)", "Iterations", "Trouve");
    for n in [8, 50, 100, 500] {
        let mut rng = Mt19937::seeded(42);
        let start = Instant::now();
        let (solution, steps, _history) = min_conflicts(n, 10 * n, &mut rng);
        let ms = start.elapsed().as_secs_f64() * 1_000.0;
        let found = if solution.is_some() { "Oui" } else { "Non" };
        println!("{n:>6}  {ms:>12.1}  {steps:>12}  {found:>8}");
    }
};


     N    Temps (ms)    Iterations    Trouve


     8           0.0            15       Oui


    50           3.6           152       Oui


   100           7.0           115       Oui


   500         475.2           346       Oui


Observons aussi la convergence des conflits au fil des itérations, relevée toutes les 20 itérations, sur une instance de taille $N = 100$.

In [13]:
// Courbe de convergence : evolution des conflits au fil des iterations (N=100)
{
    struct Mt19937 { state: [u32; 624], index: usize }

    impl Mt19937 {
        fn seeded(seed: u64) -> Self {
            // Initialisation par tableau de mots de 32 bits, comme CPython.
            let key = [seed as u32, (seed >> 32) as u32];
            let key_len = if key[1] == 0 { 1 } else { 2 };
            let mut state = [0u32; 624];
            state[0] = 19_650_218;
            for i in 1..624 {
                let p = state[i - 1];
                state[i] = 1_812_433_253u32.wrapping_mul(p ^ (p >> 30)).wrapping_add(i as u32);
            }
            let (mut i, mut j) = (1, 0);
            for _ in 0..624 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_664_525))
                    .wrapping_add(key[j]).wrapping_add(j as u32);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
                j = (j + 1) % key_len;
            }
            for _ in 0..623 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_566_083_941))
                    .wrapping_sub(i as u32);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            state[0] = 0x8000_0000;
            Self { state, index: 624 }
        }

        fn next_u32(&mut self) -> u32 {
            if self.index == 624 {
                for i in 0..624 {
                    let bits = (self.state[i] & 0x8000_0000)
                        | (self.state[(i + 1) % 624] & 0x7fff_ffff);
                    let odd_mask = if bits & 1 == 0 { 0 } else { 0x9908_b0df };
                    self.state[i] = self.state[(i + 397) % 624] ^ (bits >> 1) ^ odd_mask;
                }
                self.index = 0;
            }
            let mut bits = self.state[self.index];
            self.index += 1;
            bits ^= bits >> 11;
            bits ^= (bits << 7) & 0x9d2c_5680;
            bits ^= (bits << 15) & 0xefc6_0000;
            bits ^ (bits >> 18)
        }

        fn randbelow(&mut self, n: usize) -> usize {
            assert!(n > 0 && n <= u32::MAX as usize);
            let n = n as u32;
            // Tirage par rejet, même pour les longueurs qui sont des puissances de deux.
            loop {
                let r = self.next_u32() >> n.leading_zeros();
                if r < n { return r as usize; }
            }
        }

        fn shuffle<T>(&mut self, values: &mut [T]) {
            for i in (1..values.len()).rev() { values.swap(i, self.randbelow(i + 1)); }
        }

        fn choice<'a, T>(&mut self, values: &'a [T]) -> &'a T {
            &values[self.randbelow(values.len())]
        }
    }

    fn conflicts_at(queens: &[usize], col: usize, row: usize) -> usize {
        queens.iter().enumerate().filter(|&(c, &r)| {
            c != col && (r == row || c.abs_diff(col) == r.abs_diff(row))
        }).count()
    }

    fn total_conflicts(queens: &[usize]) -> usize {
        let mut total = 0;
        for (col, &row) in queens.iter().enumerate() {
            for (c, &r) in queens.iter().enumerate().skip(col + 1) {
                if r == row || c - col == r.abs_diff(row) { total += 1; }
            }
        }
        total
    }

    fn min_conflicts(n: usize, max_steps: usize, rng: &mut Mt19937)
        -> (Option<Vec<usize>>, usize, Vec<usize>)
    {
        let mut queens: Vec<_> = (0..n).collect();
        rng.shuffle(&mut queens);
        // Le premier point précède toute réparation ; ensuite on note un point par pas.
        let mut history = vec![total_conflicts(&queens)];
        for step in 0..max_steps {
            let conflicted: Vec<_> = (0..n)
                .filter(|&c| conflicts_at(&queens, c, queens[c]) > 0).collect();
            if conflicted.is_empty() { return (Some(queens), step, history); }
            let col = *rng.choice(&conflicted);
            let mut minimum = usize::MAX;
            let mut best_rows = Vec::new();
            for row in 0..n {
                let conflicts = conflicts_at(&queens, col, row);
                if conflicts < minimum {
                    minimum = conflicts;
                    best_rows.clear();
                }
                if conflicts == minimum { best_rows.push(row); }
            }
            queens[col] = *rng.choice(&best_rows);
            history.push(total_conflicts(&queens));
        }
        (None, max_steps, history)
    }

    let mut rng = Mt19937::seeded(123);
    let (solution, steps, history) = min_conflicts(100, 2_000, &mut rng);
    if let Some(ref queens) = solution {
        assert_eq!(total_conflicts(queens), 0);
        println!("Convergence Min-Conflicts (N=100, resolu en {steps} iterations)");
    } else {
        println!("Convergence Min-Conflicts (N=100, non resolu apres {steps} iterations)");
    }
    assert_eq!(history.len(), steps + 1);
    println!("Historique : {} points, {} conflits initiaux, {} conflits finaux",
        history.len(), history[0], history[steps]);
    println!("{:>10}  {:>10}", "Iteration", "Conflits");
    for (step, conflicts) in history.iter().enumerate() {
        if step % 20 == 0 || step == steps {
            println!("{step:>10}  {conflicts:>10}");
        }
    }
};


Convergence Min-Conflicts (N=100, resolu en 110 iterations)


Historique : 111 points, 70 conflits initiaux, 0 conflits finaux


 Iteration    Conflits


         0          70


        20          44


        40          29


        60          14


        80           7


       100           2


       110           0


### Interpretation : scalabilite de Min-Conflicts

**Sortie obtenue** : min-conflicts resout le probleme des N-Reines sur des instances jusqu'a $N = 500$ en un nombre d'iterations **non exponentiel** (le temps machine d'execution depend de la machine ; voir note ci-dessous).

| N | Temps min-conflicts (machine-dep) | Iterations | Trouve |
|---|----------------------------------|-----------|--------|
| 8   | *machine-dep* (cellule de calcul ci-dessus) | 15  | Oui |
| 50  | *machine-dep* (cellule de calcul ci-dessus) | 152 | Oui |
| 100 | *machine-dep* (cellule de calcul ci-dessus) | 115 | Oui |
| 500 | *machine-dep* (cellule de calcul ci-dessus) | 346 | Oui |

> **Note methodologique -- separation structurel / machine-dep** : les colonnes **$N$**, **Iterations** et **Trouve** sont des invariants structurels pour `Mt19937::seeded(42)` (cf. cellule de calcul ci-dessus, ligne `Mt19937::seeded(42)`). En revanche, les **temps d'execution** dependent de la machine (CPU, charge systeme, version de Rust, niveau d'optimisation, thermal throttling) et **ne survivent pas** a une re-execution sur une autre machine -- d'ou le label `*machine-dep*`. Le tableau preserve l'exacte these pedagogique (les iterations ne croissent pas exponentiellement) sans encher les runtimes volatils. La cellule de calcul du tableau reste executable localement : l'etudiant peut y observer ses propres timings sur sa machine.

**Points cles** :
1. Le nombre d'iterations croit avec $N$ (15 -> 152 -> 115 -> 346), mais de maniere irreguliere ; il ne croit pas exponentiellement
2. Le tableau de convergence montre une decroissance rapide des conflits
3. Le temps par iteration est $O(N^2)$ (compter les conflits) ; sur la plus grande instance committee (N=500), le temps total atteint un ordre de grandeur appreciable (runtime *machine-dep*, voir cellule de calcul), avec une croissance super-quadratique en pratique. Le **ratio** entre (par exemple) N=500 et N=100 derive des deux timings machine-dep et n'est donc pas un invariant reproductible d'une machine a l'autre.

> **Pourquoi ca marche ?** L'espace des solutions des N-Reines est "dense" pour les instances aleatoires. En partant d'une permutation aleatoire, on est generalement proche d'une solution. Chaque reparation locale reduit les conflits sans en creer beaucoup de nouveaux.

> **Attention** : min-conflicts est **incomplet** -- il ne garantit pas de trouver une solution et ne peut pas prouver qu'il n'en existe pas.

***

## 5. Approche 3 : OR-Tools CP-SAT

**OR-Tools CP-SAT** est le solveur de programmation par contraintes de Google. Il combine propagation de contraintes, recherche avec apprentissage de clauses (clause learning), et parallelisme.

### Avantages d'un solveur industriel

| Fonctionnalite | Implementation manuelle | CP-SAT |
|----------------|------------------------|--------|
| Propagation | Forward Checking basique | AC, bounds consistency, specialisee |
| Apprentissage | Aucun | Clause learning (CDCL) |
| Parallelisme | Non | Multi-thread natif |
| Contraintes globales | A implementer | `add_all_different`, etc. |
| Enumeration | A implementer | `enumerate_all_solutions` |

In [14]:
// Modéliser les N-Reines avec CP-SAT
{
    use cp_sat::builder::{CpModelBuilder, LinearExpr};
    use cp_sat::proto::{CpSolverStatus, SatParameters};
    use std::time::Instant;

    fn solve_nqueens_cpsat(n: usize, enumerate_all: bool, time_limit_s: f64)
        -> (Vec<Vec<usize>>, f64, CpSolverStatus)
    {
        assert!(n > 0);
        let mut model = CpModelBuilder::default();
        // Variables : queens[i] = ligne de la reine en colonne i
        let queens: Vec<_> = (0..n).map(|_| model.new_int_var(std::iter::once((0i64, n as i64 - 1)))).collect();
        // Contrainte 1 : toutes les lignes differentes
        model.add_all_different(queens.iter().copied());
        // Contrainte 2 : toutes les diagonales montantes differentes
        // queens[i] + i doit etre different pour tout i
        model.add_all_different(queens.iter().copied().enumerate()
            .map(|(i, q)| LinearExpr::from(q) + i as i64));
        // Contrainte 3 : toutes les diagonales descendantes differentes
        // queens[i] - i doit etre different pour tout i
        model.add_all_different(queens.iter().copied().enumerate()
            .map(|(i, q)| LinearExpr::from(q) + -(i as i64)));

        // Callback pour collecter toutes les solutions : le collecteur Python est remplacé
        // par le réservoir natif, dimensionné pour les n! permutations possibles pour N <= 8.
        let mut params = SatParameters { max_time_in_seconds: Some(time_limit_s), ..Default::default() };
        if enumerate_all {
            assert!(n <= 8, "Cette collecte exhaustive est dimensionnée pour N <= 8");
            params.enumerate_all_solutions = Some(true);
            params.num_search_workers = Some(1);
            params.fill_additional_solutions_in_response = Some(true);
            params.solution_pool_size = Some((1..=n as i32).product());
        }
        let start = Instant::now();
        let response = model.solve_with_parameters(&params);
        let ms = start.elapsed().as_secs_f64() * 1_000.0;
        let status = response.status();
        // Les N premières variables du modèle sont les reines, dans l'ordre des colonnes.
        let placement = |values: &[i64]| values[..n].iter().map(|&r| r as usize).collect::<Vec<_>>();
        let solutions = if enumerate_all {
            response.additional_solutions.iter().map(|s| placement(&s.values)).collect()
        } else {
            match status {
                CpSolverStatus::Optimal | CpSolverStatus::Feasible => vec![placement(&response.solution)],
                _ => Vec::new(),
            }
        };
        (solutions, ms, status)
    }

    // Resoudre N=8
    let (solutions, ms, status) = solve_nqueens_cpsat(8, false, 30.0);
    println!("CP-SAT N=8 : {}", format!("{status:?}").to_ascii_uppercase());
    match solutions.first() {
        Some(q) => println!("Solution    : {q:?}"),
        None => println!("Solution    : Aucune"),
    }
    println!("Temps       : {ms:.1} ms");
};


CP-SAT N=8 : OPTIMAL


Solution    : [4, 7, 3, 0, 6, 1, 5, 2]


Temps       : 24.1 ms


### Enumeration de toutes les solutions (N=8)

Un avantage majeur de CP-SAT est sa capacite a **enumerer toutes les solutions**. Pour $N = 8$, on sait qu'il en existe exactement 92.

In [15]:
// Enumerer toutes les solutions pour N=8
{
    use cp_sat::builder::{CpModelBuilder, LinearExpr};
    use cp_sat::proto::{CpSolverStatus, SatParameters};
    use std::collections::BTreeSet;
    use std::time::Instant;

    fn enumerate_nqueens_cpsat(n: usize, time_limit_s: f64)
        -> (Vec<Vec<usize>>, f64, CpSolverStatus)
    {
        assert!((1..=8).contains(&n));
        let mut model = CpModelBuilder::default();
        let queens: Vec<_> = (0..n).map(|_| model.new_int_var(std::iter::once((0i64, n as i64 - 1)))).collect();
        // Une contrainte globale pour les lignes, puis une pour chaque famille de diagonales.
        model.add_all_different(queens.iter().copied());
        model.add_all_different(queens.iter().copied().enumerate()
            .map(|(i, q)| LinearExpr::from(q) + i as i64));
        model.add_all_different(queens.iter().copied().enumerate()
            .map(|(i, q)| LinearExpr::from(q) + -(i as i64)));

        // L'énumération native est séquentielle. Le réservoir couvre au moins
        // n! placements, borne supérieure donnée par AllDifferent sur les lignes.
        let params = SatParameters {
            max_time_in_seconds: Some(time_limit_s),
            num_search_workers: Some(1),
            enumerate_all_solutions: Some(true),
            fill_additional_solutions_in_response: Some(true),
            solution_pool_size: Some((1..=n as i32).product()),
            ..Default::default()
        };
        let start = Instant::now();
        let response = model.solve_with_parameters(&params);
        let ms = start.elapsed().as_secs_f64() * 1_000.0;
        let mut seen = BTreeSet::new();
        let mut solutions = Vec::new();
        // Préserver l'ordre de la réponse et ne pas compter deux fois le placement principal.
        for values in response.additional_solutions.iter().map(|s| s.values.as_slice())
            .chain(std::iter::once(response.solution.as_slice()))
        {
            if values.len() >= n {
                let q: Vec<_> = values[..n].iter().map(|&row| row as usize).collect();
                if seen.insert(q.clone()) { solutions.push(q); }
            }
        }
        (solutions, ms, response.status())
    }

    fn draw_queens_text(queens: &[usize], n: usize, title: &str) {
        println!("{title}");
        for row in 0..n {
            let line: String = queens.iter().enumerate().map(|(col, &r)| {
                if r == row { '♛' } else if (row + col) % 2 == 0 { '.' } else { '#' }
            }).collect();
            println!("{line}");
        }
    }

    let n = 8;
    let (solutions, ms, status) = enumerate_nqueens_cpsat(n, 30.0);
    println!("N={n} : {} solutions trouvees en {ms:.1} ms", solutions.len());
    println!("Status : {}", format!("{status:?}").to_ascii_uppercase());
    assert_eq!(status, CpSolverStatus::Optimal);
    assert_eq!(solutions.len(), 92);

    // Afficher les 4 premieres solutions
    for (i, queens) in solutions.iter().take(4).enumerate() {
        draw_queens_text(queens, n, &format!("Solution {}", i + 1));
    }
};


N=8 : 92 solutions trouvees en 67.6 ms


Status : OPTIMAL


Solution 1


♛#.#.#.#


#.#.#.♛.


.#.#♛#.#


... (28 outputs left out)

.#♛#.#.#


#.#.#.♛.


.♛.#.#.#


#.#♛#.#.


### Performance CP-SAT sur de grandes instances

Testons CP-SAT sur des tailles croissantes, jusqu'a $N = 500$.

In [16]:
// Benchmark CP-SAT sur differentes tailles
{
    use cp_sat::builder::{CpModelBuilder, LinearExpr};
    use cp_sat::proto::{CpSolverStatus, SatParameters};
    use std::time::Instant;

    fn solve_nqueens_cpsat(n: usize, time_limit_s: f64) -> (f64, CpSolverStatus) {
        let mut model = CpModelBuilder::default();
        let queens: Vec<_> = (0..n).map(|_| model.new_int_var(std::iter::once((0i64, n as i64 - 1)))).collect();
        // Les lignes et les deux familles de diagonales sont distinctes.
        model.add_all_different(queens.iter().copied());
        model.add_all_different(queens.iter().copied().enumerate()
            .map(|(i, q)| LinearExpr::from(q) + i as i64));
        model.add_all_different(queens.iter().copied().enumerate()
            .map(|(i, q)| LinearExpr::from(q) + -(i as i64)));
        let params = SatParameters { max_time_in_seconds: Some(time_limit_s), ..Default::default() };
        // Chronométrer l'appel au solveur, sans la construction du modèle.
        let start = Instant::now();
        let response = model.solve_with_parameters(&params);
        (start.elapsed().as_secs_f64() * 1_000.0, response.status())
    }

    fn status_label(status: CpSolverStatus) -> &'static str {
        match status {
            CpSolverStatus::Optimal => "OPTIMAL",
            CpSolverStatus::Feasible => "FEASIBLE",
            CpSolverStatus::Infeasible => "INFEASIBLE",
            CpSolverStatus::Unknown => "UNKNOWN",
            CpSolverStatus::ModelInvalid => "MODEL_INVALID",
        }
    }

    println!("Limite CP-SAT : 60 s par taille");
    println!("{:>6}  {:>12}  {:>13}", "N", "Temps (ms)", "Status");
    println!("{}", "-".repeat(35));
    for n in [8, 50, 100, 500] {
        let (ms, status) = solve_nqueens_cpsat(n, 60.0);
        println!("{n:>6}  {ms:>12.1}  {:>13}", status_label(status));
    }
};


Limite CP-SAT : 60 s par taille


     N    Temps (ms)         Status


-----------------------------------


     8          11.6        OPTIMAL


    50         298.8        OPTIMAL


   100        1657.2        OPTIMAL


   500       60035.1        UNKNOWN


### Interprétation : OR-Tools CP-SAT

Dans l’exécution présentée, CP-SAT trouve une solution pour les trois premières tailles du benchmark. Pour $N = 500$, il atteint la limite de 60 s et renvoie `UNKNOWN` : il n’a ni fourni de solution ni établi qu’il n’en existe pas dans le délai imparti.

| Fonctionnalité | Résultat |
|----------------|----------|
| Première solution ($N = 8$) | Temps mesuré dans l’exemple de résolution |
| Toutes les solutions ($N = 8$) | Exactement 92 solutions distinctes |
| Résolution sous limite de temps ($N = 500$) | `UNKNOWN` dans l’exécution présentée |

> Le nombre de solutions pour $N = 8$ est une propriété combinatoire. En revanche, les temps mesurés et le statut obtenu sous une limite de temps dépendent de la machine et de la configuration du solveur. La même limite de 60 s ne garantit donc pas le même statut à chaque exécution.

**Avantages de CP-SAT** :
1. **Modélisation déclarative** : on décrit les contraintes, le solveur choisit la stratégie.
2. **`add_all_different`** : une contrainte globale exprime directement que les valeurs doivent être distinctes.
3. **Énumération complète** : le solveur peut rechercher toutes les solutions.
4. **Robustesse** : il n’est pas nécessaire de régler les heuristiques à la main.

> **Compromis** : la préparation du modèle a un coût. Lorsque seule une solution est nécessaire, min-conflicts peut être plus rapide sur de grandes instances, sans pour autant garantir qu’il en trouvera une.

***

## 6. Comparaison et analyse

Consolidons les résultats des trois approches dans un benchmark comparatif.

In [17]:
// Benchmark comparatif sur les tailles communes
{
    use cp_sat::builder::{CpModelBuilder, LinearExpr};
    use cp_sat::proto::{CpSolverStatus, SatParameters};
    use std::time::Instant;

    fn backtracking_simple(n: usize) -> Option<(Vec<usize>, usize)> {
        fn solve(q: &mut [usize], col: usize, nodes: &mut usize) -> bool {
            if col == q.len() { return true; }
            for row in 0..q.len() {
                *nodes += 1;
                if q[..col].iter().enumerate().all(|(c, &r)| r != row && col - c != r.abs_diff(row)) {
                    q[col] = row;
                    if solve(q, col + 1, nodes) { return true; }
                }
            }
            false
        }
        let mut q = vec![0; n];
        let mut nodes = 0;
        let found = solve(&mut q, 0, &mut nodes);
        found.then_some((q, nodes))
    }

    // Reprendre l'ordre des domaines CPython de petits entiers : emplacements
    // parcourus dans l'ordre, marques de suppression et copies compactes.
    #[derive(Clone, Copy, PartialEq, Eq)]
    enum Slot { Empty, Deleted, Row(usize) }
    #[derive(Clone)]
    struct Domain { slots: Vec<Slot>, len: usize, fill: usize }

    impl Domain {
        fn empty() -> Self { Self { slots: vec![Slot::Empty; 8], len: 0, fill: 0 } }
        fn rows(&self) -> impl Iterator<Item = usize> + '_ {
            self.slots.iter().filter_map(|s| match *s { Slot::Row(r) => Some(r), _ => None })
        }
        fn locate(&self, row: usize) -> Result<usize, usize> {
            let mask = self.slots.len() - 1;
            let (mut i, mut perturb, mut deleted) = (row & mask, row, None);
            loop {
                let end = if i + 9 <= mask { i + 9 } else { i };
                for j in i..=end {
                    match self.slots[j] {
                        Slot::Row(r) if r == row => return Ok(j),
                        Slot::Empty => return Err(deleted.unwrap_or(j)),
                        Slot::Deleted => deleted = Some(j),
                        _ => {}
                    }
                }
                perturb >>= 5;
                i = i.wrapping_mul(5).wrapping_add(1).wrapping_add(perturb) & mask;
            }
        }
        fn insert_without_resize(&mut self, row: usize) {
            if let Err(i) = self.locate(row) {
                if self.slots[i] == Slot::Empty { self.fill += 1; }
                self.slots[i] = Slot::Row(row);
                self.len += 1;
            }
        }
        fn table_size(min_used: usize) -> usize {
            let mut size = 8;
            while size <= min_used { size *= 2; }
            size
        }
        fn rehash(&self, min_used: usize) -> Self {
            let mut out = Self { slots: vec![Slot::Empty; Self::table_size(min_used)], len: 0, fill: 0 };
            for row in self.rows() { out.insert_without_resize(row); }
            out
        }
        fn insert(&mut self, row: usize) {
            let old_fill = self.fill;
            self.insert_without_resize(row);
            if self.fill != old_fill && self.fill * 5 >= (self.slots.len() - 1) * 3 {
                *self = self.rehash(self.len * 4);
            }
        }
        fn remove(&mut self, row: usize) -> bool {
            if let Ok(i) = self.locate(row) {
                self.slots[i] = Slot::Deleted;
                self.len -= 1;
                true
            } else {
                false
            }
        }
        fn copy_domain(&self) -> Self {
            let min_used = if self.len * 5 >= 7 * 3 { self.len * 2 } else { 0 };
            if Self::table_size(min_used) == self.slots.len() && self.fill == self.len {
                self.clone()
            } else {
                self.rehash(min_used)
            }
        }
    }

    fn backtracking_fc(n: usize) -> Option<(Vec<usize>, usize)> {
        fn solve(q: &mut [Option<usize>], d: &mut [Domain], nodes: &mut usize) -> bool {
            let Some(col) = (0..q.len()).filter(|&c| q[c].is_none())
                .min_by_key(|&c| (d[c].len, c)) else { return true; };
            let rows: Vec<_> = d[col].rows().collect();
            for row in rows {
                *nodes += 1;
                q[col] = Some(row);
                let old = d[col].copy_domain();
                d[col] = Domain::empty();
                let mut pruned = Vec::new();
                for c in 0..q.len() {
                    if q[c].is_some() { continue; }
                    let gap = c.abs_diff(col);
                    for r in [Some(row), row.checked_sub(gap), row.checked_add(gap)].iter().copied().flatten() {
                        if r < q.len() && d[c].remove(r) { pruned.push((c, r)); }
                    }
                }
                let empty = (0..q.len()).any(|c| q[c].is_none() && d[c].len == 0);
                if !empty && solve(q, d, nodes) { return true; }
                for (c, r) in pruned { d[c].insert(r); }
                d[col] = old;
                q[col] = None;
            }
            false
        }
        let mut full = Domain::empty();
        for r in 0..n { full.insert(r); }
        let mut d = vec![full; n];
        let mut q = vec![None; n];
        let mut nodes = 0;
        if solve(&mut q, &mut d, &mut nodes) {
            q.into_iter().collect::<Option<Vec<_>>>().map(|rows| (rows, nodes))
        } else {
            None
        }
    }

    struct Mt19937 { state: [u32; 624], index: usize }
    impl Mt19937 {
        fn seeded(seed: u32) -> Self {
            // Initialisation CPython pour les graines tenant sur un mot.
            let mut state = [0u32; 624];
            state[0] = 19_650_218;
            for i in 1..624 {
                let p = state[i - 1];
                state[i] = 1_812_433_253u32.wrapping_mul(p ^ (p >> 30)).wrapping_add(i as u32);
            }
            let mut i = 1;
            for _ in 0..624 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_664_525)).wrapping_add(seed);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            for _ in 0..623 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_566_083_941)).wrapping_sub(i as u32);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            state[0] = 0x8000_0000;
            Self { state, index: 624 }
        }
        fn next_u32(&mut self) -> u32 {
            if self.index == 624 {
                for i in 0..624 {
                    let bits = (self.state[i] & 0x8000_0000) | (self.state[(i + 1) % 624] & 0x7fff_ffff);
                    let odd = if bits & 1 == 0 { 0 } else { 0x9908_b0df };
                    self.state[i] = self.state[(i + 397) % 624] ^ (bits >> 1) ^ odd;
                }
                self.index = 0;
            }
            let mut bits = self.state[self.index];
            self.index += 1;
            bits ^= bits >> 11;
            bits ^= (bits << 7) & 0x9d2c_5680;
            bits ^= (bits << 15) & 0xefc6_0000;
            bits ^ (bits >> 18)
        }
        fn randbelow(&mut self, n: usize) -> usize {
            assert!(n > 0 && n <= u32::MAX as usize);
            let n = n as u32;
            loop {
                let r = self.next_u32() >> n.leading_zeros();
                if r < n { return r as usize; }
            }
        }
    }

    fn conflicts_at(q: &[usize], col: usize, row: usize) -> usize {
        q.iter().enumerate().filter(|&(c, &r)| c != col && (r == row || c.abs_diff(col) == r.abs_diff(row))).count()
    }
    fn total_conflicts(q: &[usize]) -> usize {
        let mut count = 0;
        for (c, &r) in q.iter().enumerate() {
            for (other, &y) in q.iter().enumerate().skip(c + 1) {
                if r == y || other - c == r.abs_diff(y) { count += 1; }
            }
        }
        count
    }
    fn min_conflicts(n: usize, rng: &mut Mt19937) -> (Option<Vec<usize>>, usize, Vec<usize>) {
        let mut q: Vec<_> = (0..n).collect();
        // Fisher–Yates inversé, puis choix par indice parmi les candidats ordonnés.
        for i in (1..n).rev() { q.swap(i, rng.randbelow(i + 1)); }
        let max_steps = (5 * n).max(1_000);
        let mut history = vec![total_conflicts(&q)];
        for step in 0..max_steps {
            let conflicted: Vec<_> = (0..n).filter(|&c| conflicts_at(&q, c, q[c]) > 0).collect();
            if conflicted.is_empty() { return (Some(q), step, history); }
            let col = conflicted[rng.randbelow(conflicted.len())];
            let mut minimum = usize::MAX;
            let mut best = Vec::new();
            for row in 0..n {
                let score = conflicts_at(&q, col, row);
                if score < minimum { minimum = score; best.clear(); }
                if score == minimum { best.push(row); }
            }
            q[col] = best[rng.randbelow(best.len())];
            history.push(total_conflicts(&q));
        }
        (None, max_steps, history)
    }

    fn solve_nqueens_cpsat(n: usize) -> (f64, CpSolverStatus) {
        let mut model = CpModelBuilder::default();
        let q: Vec<_> = (0..n).map(|_| model.new_int_var(std::iter::once((0i64, n as i64 - 1)))).collect();
        model.add_all_different(q.iter().copied());
        model.add_all_different(q.iter().copied().enumerate().map(|(i, v)| LinearExpr::from(v) + i as i64));
        model.add_all_different(q.iter().copied().enumerate().map(|(i, v)| LinearExpr::from(v) + -(i as i64)));
        let params = SatParameters { max_time_in_seconds: Some(30.0), ..Default::default() };
        let start = Instant::now();
        let response = model.solve_with_parameters(&params);
        (start.elapsed().as_secs_f64() * 1_000.0, response.status())
    }
    fn ms_text(ms: Option<f64>) -> String {
        ms.map_or_else(|| "--".to_owned(), |t| format!("{t:.3}"))
    }

    println!("Benchmark comparatif N-Reines");
    println!("{:>4}  {:>12}  {:>12}  {:>12}  {:>12}   (ms)", "N", "BT Simple", "BT FC+MRV", "Min-Conf.", "CP-SAT");
    for n in [8, 50, 100] {
        // Backtracking simple (limite a N <= 20 pour eviter des temps trop longs)
        let simple = (n <= 20).then(|| {
            let start = Instant::now();
            let result = backtracking_simple(n);
            let ms = start.elapsed().as_secs_f64() * 1_000.0;
            assert!(result.is_some());
            ms
        });
        // Backtracking FC+MRV (limite a N <= 50)
        let fc = (n <= 50).then(|| {
            let start = Instant::now();
            let result = backtracking_fc(n);
            let ms = start.elapsed().as_secs_f64() * 1_000.0;
            assert!(result.is_some());
            ms
        });
        // Min-Conflicts
        let mut rng = Mt19937::seeded(42);
        let start = Instant::now();
        let (solution, _, _) = min_conflicts(n, &mut rng);
        let mc = start.elapsed().as_secs_f64() * 1_000.0;
        assert!(solution.is_some());
        // CP-SAT
        let (cp, status) = solve_nqueens_cpsat(n);
        println!("{n:>4}  {:>12}  {:>12}  {mc:>12.3}  {cp:>12.3}   {}", ms_text(simple), ms_text(fc),
            format!("{status:?}").to_ascii_uppercase());
    }
    println!("(-- = mesure omise pour cette taille)");
};


Benchmark comparatif N-Reines


   N     BT Simple     BT FC+MRV     Min-Conf.        CP-SAT   (ms)


   8         0.017         0.124         0.030        14.164   OPTIMAL


  50            --         3.611         2.195       256.754   OPTIMAL


 100            --            --         8.073      1687.668   OPTIMAL


(-- = mesure omise pour cette taille)


Comparons les temps d'exécution sur une plage etendue de tailles, approche par approche, dans un tableau.

In [18]:
// Graphique comparatif sur les tailles gerees par toutes les approches
{
    use cp_sat::builder::{CpModelBuilder, LinearExpr};
    use cp_sat::proto::{CpSolverStatus, SatParameters};
    use std::collections::BTreeMap;
    use std::time::Instant;

    struct Mt19937 { state: [u32; 624], index: usize }
    impl Mt19937 {
        fn seeded(seed: u32) -> Self {
            // Initialisation CPython par tableau d'un mot de 32 bits.
            let mut state = [0u32; 624];
            state[0] = 19_650_218;
            for i in 1..624 {
                let p = state[i - 1];
                state[i] = 1_812_433_253u32.wrapping_mul(p ^ (p >> 30)).wrapping_add(i as u32);
            }
            let mut i = 1;
            for _ in 0..624 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_664_525)).wrapping_add(seed);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            for _ in 0..623 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_566_083_941)).wrapping_sub(i as u32);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            state[0] = 0x8000_0000;
            Self { state, index: 624 }
        }
        fn next_u32(&mut self) -> u32 {
            if self.index == 624 {
                for i in 0..624 {
                    let bits = (self.state[i] & 0x8000_0000) | (self.state[(i + 1) % 624] & 0x7fff_ffff);
                    let odd = if bits & 1 == 0 { 0 } else { 0x9908_b0df };
                    self.state[i] = self.state[(i + 397) % 624] ^ (bits >> 1) ^ odd;
                }
                self.index = 0;
            }
            let mut bits = self.state[self.index];
            self.index += 1;
            bits ^= bits >> 11;
            bits ^= (bits << 7) & 0x9d2c_5680;
            bits ^= (bits << 15) & 0xefc6_0000;
            bits ^ (bits >> 18)
        }
        fn randbelow(&mut self, n: usize) -> usize {
            assert!(n > 0 && n <= u32::MAX as usize);
            let n = n as u32;
            loop {
                let r = self.next_u32() >> n.leading_zeros();
                if r < n { return r as usize; }
            }
        }
    }

    fn conflicts_at(q: &[usize], col: usize, row: usize) -> usize {
        q.iter().enumerate().filter(|&(c, &r)| c != col && (r == row || c.abs_diff(col) == r.abs_diff(row))).count()
    }
    fn total_conflicts(q: &[usize]) -> usize {
        let mut count = 0;
        for (c, &r) in q.iter().enumerate() {
            for (other, &y) in q.iter().enumerate().skip(c + 1) {
                if r == y || other - c == r.abs_diff(y) { count += 1; }
            }
        }
        count
    }
    fn min_conflicts(n: usize, rng: &mut Mt19937) -> (Option<Vec<usize>>, usize, Vec<usize>) {
        let mut q: Vec<_> = (0..n).collect();
        for i in (1..n).rev() { q.swap(i, rng.randbelow(i + 1)); }
        let max_steps = (5 * n).max(1_000);
        // Conserver le coût de l'historique, comme dans le solveur comparé.
        let mut history = vec![total_conflicts(&q)];
        for step in 0..max_steps {
            let conflicted: Vec<_> = (0..n).filter(|&c| conflicts_at(&q, c, q[c]) > 0).collect();
            if conflicted.is_empty() { return (Some(q), step, history); }
            let col = conflicted[rng.randbelow(conflicted.len())];
            let mut minimum = usize::MAX;
            let mut best = Vec::new();
            for row in 0..n {
                let score = conflicts_at(&q, col, row);
                if score < minimum { minimum = score; best.clear(); }
                if score == minimum { best.push(row); }
            }
            q[col] = best[rng.randbelow(best.len())];
            history.push(total_conflicts(&q));
        }
        (None, max_steps, history)
    }

    fn solve_nqueens_cpsat(n: usize) -> (f64, CpSolverStatus) {
        let mut model = CpModelBuilder::default();
        let q: Vec<_> = (0..n).map(|_| model.new_int_var(std::iter::once((0i64, n as i64 - 1)))).collect();
        model.add_all_different(q.iter().copied());
        model.add_all_different(q.iter().copied().enumerate().map(|(i, v)| LinearExpr::from(v) + i as i64));
        model.add_all_different(q.iter().copied().enumerate().map(|(i, v)| LinearExpr::from(v) + -(i as i64)));
        let params = SatParameters { max_time_in_seconds: Some(30.0), ..Default::default() };
        let start = Instant::now();
        let response = model.solve_with_parameters(&params);
        (start.elapsed().as_secs_f64() * 1_000.0, response.status())
    }

    // Garder le même ordre de parcours et les mêmes restaurations qu'en c15.
    #[derive(Clone, Copy, PartialEq, Eq)]
    enum Slot { Empty, Deleted, Row(usize) }
    #[derive(Clone)]
    struct Domain { slots: Vec<Slot>, len: usize, fill: usize }
    impl Domain {
        fn empty() -> Self { Self { slots: vec![Slot::Empty; 8], len: 0, fill: 0 } }
        fn rows(&self) -> impl Iterator<Item = usize> + '_ {
            self.slots.iter().filter_map(|s| match *s { Slot::Row(r) => Some(r), _ => None })
        }
        fn locate(&self, row: usize) -> Result<usize, usize> {
            let mask = self.slots.len() - 1;
            let (mut i, mut perturb, mut deleted) = (row & mask, row, None);
            loop {
                let end = if i + 9 <= mask { i + 9 } else { i };
                for j in i..=end {
                    match self.slots[j] {
                        Slot::Row(r) if r == row => return Ok(j),
                        Slot::Empty => return Err(deleted.unwrap_or(j)),
                        Slot::Deleted => deleted = Some(j),
                        _ => {}
                    }
                }
                perturb >>= 5;
                i = i.wrapping_mul(5).wrapping_add(1).wrapping_add(perturb) & mask;
            }
        }
        fn insert_without_resize(&mut self, row: usize) {
            if let Err(i) = self.locate(row) {
                if self.slots[i] == Slot::Empty { self.fill += 1; }
                self.slots[i] = Slot::Row(row);
                self.len += 1;
            }
        }
        fn table_size(min_used: usize) -> usize {
            let mut size = 8;
            while size <= min_used { size *= 2; }
            size
        }
        fn rehash(&self, min_used: usize) -> Self {
            let mut out = Self { slots: vec![Slot::Empty; Self::table_size(min_used)], len: 0, fill: 0 };
            for row in self.rows() { out.insert_without_resize(row); }
            out
        }
        fn insert(&mut self, row: usize) {
            let old_fill = self.fill;
            self.insert_without_resize(row);
            if self.fill != old_fill && self.fill * 5 >= (self.slots.len() - 1) * 3 {
                *self = self.rehash(self.len * 4);
            }
        }
        fn remove(&mut self, row: usize) -> bool {
            if let Ok(i) = self.locate(row) {
                self.slots[i] = Slot::Deleted;
                self.len -= 1;
                true
            } else {
                false
            }
        }
        fn copy_domain(&self) -> Self {
            let min_used = if self.len * 5 >= 7 * 3 { self.len * 2 } else { 0 };
            if Self::table_size(min_used) == self.slots.len() && self.fill == self.len {
                self.clone()
            } else {
                self.rehash(min_used)
            }
        }
    }
    fn backtracking_fc(n: usize) -> Option<(Vec<usize>, usize)> {
        fn solve(q: &mut [Option<usize>], d: &mut [Domain], nodes: &mut usize) -> bool {
            let Some(col) = (0..q.len()).filter(|&c| q[c].is_none())
                .min_by_key(|&c| (d[c].len, c)) else { return true; };
            let rows: Vec<_> = d[col].rows().collect();
            for row in rows {
                *nodes += 1;
                q[col] = Some(row);
                let old = d[col].copy_domain();
                d[col] = Domain::empty();
                let mut pruned = Vec::new();
                for c in 0..q.len() {
                    if q[c].is_some() { continue; }
                    let gap = c.abs_diff(col);
                    for r in [Some(row), row.checked_sub(gap), row.checked_add(gap)].iter().copied().flatten() {
                        if r < q.len() && d[c].remove(r) { pruned.push((c, r)); }
                    }
                }
                let empty = (0..q.len()).any(|c| q[c].is_none() && d[c].len == 0);
                if !empty && solve(q, d, nodes) { return true; }
                for (c, r) in pruned { d[c].insert(r); }
                d[col] = old;
                q[col] = None;
            }
            false
        }
        let mut full = Domain::empty();
        for r in 0..n { full.insert(r); }
        let mut d = vec![full; n];
        let mut q = vec![None; n];
        let mut nodes = 0;
        if solve(&mut q, &mut d, &mut nodes) {
            q.into_iter().collect::<Option<Vec<_>>>().map(|rows| (rows, nodes))
        } else {
            None
        }
    }

    #[derive(Default)]
    struct Row {
        mc: Option<(f64, bool)>,
        cp: Option<(f64, CpSolverStatus)>,
        fc: Option<(f64, bool)>,
    }
    fn ms_text(ms: Option<f64>) -> String {
        ms.map_or_else(|| "--".to_owned(), |t| format!("{t:.3}"))
    }
    fn found_text(found: Option<bool>) -> &'static str {
        match found { Some(true) => "Oui", Some(false) => "Non", None => "--" }
    }

    let mut rows = BTreeMap::<usize, Row>::new();
    // Min-Conflicts et CP-SAT sur toute la plage
    for n in [8, 50, 100, 500] {
        let mut rng = Mt19937::seeded(42);
        let start = Instant::now();
        let (solution, _, _) = min_conflicts(n, &mut rng);
        let ms = start.elapsed().as_secs_f64() * 1_000.0;
        let row = rows.entry(n).or_default();
        row.mc = Some((ms, solution.is_some()));
        row.cp = Some(solve_nqueens_cpsat(n));
    }
    // Backtracking FC+MRV (petites tailles uniquement)
    for n in [8, 12, 16, 20] {
        let start = Instant::now();
        let solution = backtracking_fc(n);
        let ms = start.elapsed().as_secs_f64() * 1_000.0;
        rows.entry(n).or_default().fc = Some((ms, solution.is_some()));
    }

    println!("{:>4}  {:>12} {:>7}  {:>12} {:>13}  {:>12} {:>7}",
        "N", "MC (ms)", "Trouve", "CP-SAT (ms)", "Statut", "FC+MRV (ms)", "Trouve");
    for (n, row) in rows {
        let cp_status = row.cp.as_ref().map_or_else(|| "--".to_owned(), |(_, status)| {
            format!("{status:?}").to_ascii_uppercase()
        });
        println!("{n:>4}  {:>12} {:>7}  {:>12} {cp_status:>13}  {:>12} {:>7}",
            ms_text(row.mc.map(|(ms, _)| ms)), found_text(row.mc.map(|(_, ok)| ok)),
            ms_text(row.cp.map(|(ms, _)| ms)),
            ms_text(row.fc.map(|(ms, _)| ms)), found_text(row.fc.map(|(_, ok)| ok)));
    }
    println!("(-- = mesure non effectuée)");
};


   N       MC (ms)  Trouve   CP-SAT (ms)        Statut   FC+MRV (ms)  Trouve


   8         0.038     Oui         9.859       OPTIMAL         0.077     Oui


  12            --      --            --            --         0.184     Oui


  16            --      --            --            --         0.060     Oui


  20            --      --            --            --         0.206     Oui


  50         2.128     Oui       270.578       OPTIMAL            --      --


 100         9.719     Oui      1522.471       OPTIMAL            --      --


 500       482.802     Oui     30027.381       UNKNOWN            --      --


(-- = mesure non effectuée)


### Interpretation : comparaison finale

**Sortie obtenue** : les trois approches ont des profils de performance très différents.

| Approche | N=8 | N=100 | N=1000 | Completude | Enumeration |
|----------|-----|-------|--------|------------|-------------|
| Backtracking FC+MRV | Rapide | Lent | Intraitable | Oui | Oui (lent) |
| Min-Conflicts | Rapide | Rapide | Rapide | Non | Non |
| CP-SAT | Rapide | Rapide | Rapide | Oui | Oui |

### Quand utiliser quelle approche ?

| Situation | Approche recommandee | Raison |
|-----------|---------------------|--------|
| Petit problème (N < 20) | Backtracking FC+MRV | Complet, simple a implementer |
| Grande instance, une solution | Min-Conflicts | Quasi-lineaire, très rapide |
| Toutes les solutions | CP-SAT | Enumeration + optimisation |
| Contraintes supplementaires | CP-SAT | Modelisation declarative flexible |
| Prototype pédagogique | Backtracking | Comprendre les mécanismes |

### Liens avec les Foundations

| Concept applique | Notebook de reference |
|-----------------|----------------------|
| Backtracking CSP, MRV, LCV | [CSP-1 : Fundamentals](../Part2-CSP/CSP-01-Fundamentals-Python.ipynb) |
| Forward Checking, Arc Consistency | [CSP-2 : Consistency](../Part2-CSP/CSP-2-Consistency.ipynb) |
| CP-SAT, contraintes globales | [CSP-3 : Advanced](../Part2-CSP/CSP-3-Advanced.ipynb) |

***

## 7. Exemple guide

### Exemple guide 1 : le problème des N-Tours

**Enonce** : adaptez le problème pour placer $N$ **tours** sur un echiquier $N \times N$. Les tours attaquent en ligne et en colonne (mais pas en diagonale).

1. Quelle contrainte disparait par rapport aux N-Reines ?
2. Combien de solutions existe-t-il pour $N = 8$ ?
3. Implementez un solveur et verifiez votre reponse.

In [19]:
// Exemple resolu : N-Tours via CP-SAT
{
    use cp_sat::builder::CpModelBuilder;
    use cp_sat::proto::{CpSolverStatus, SatParameters};
    use std::collections::BTreeSet;
    use std::time::{Duration, Instant};

    fn count_nrooks_solutions_cpsat(n: usize, time_limit: Duration) -> (usize, CpSolverStatus, f64) {
        assert!((2..=8).contains(&n));
        let start = Instant::now();
        let mut count = 0;
        // Fixer les lignes des deux premières tours découpe les placements en
        // sous-problèmes disjoints. Chacun a son propre réservoir de (N-2)! places.
        let pool_size: i32 = (1..=(n as i32 - 2)).product();
        for first in 0..n as i64 {
            for second in 0..n as i64 {
                if first == second { continue; }
                let mut model = CpModelBuilder::default();
                let rooks: Vec<_> = (0..n).map(|col| {
                    let bounds = match col {
                        0 => (first, first),
                        1 => (second, second),
                        _ => (0, n as i64 - 1),
                    };
                    model.new_int_var(std::iter::once(bounds))
                }).collect();
                // Une tour par colonne par construction ; il suffit de distinguer les lignes.
                model.add_all_different(rooks.iter().copied());

                let remaining = time_limit.saturating_sub(start.elapsed());
                if remaining.is_zero() {
                    let status = if count == 0 { CpSolverStatus::Unknown } else { CpSolverStatus::Feasible };
                    return (count, status, start.elapsed().as_secs_f64() * 1_000.0);
                }
                let params = SatParameters {
                    max_time_in_seconds: Some(remaining.as_secs_f64()),
                    num_search_workers: Some(1),
                    enumerate_all_solutions: Some(true),
                    fill_additional_solutions_in_response: Some(true),
                    solution_pool_size: Some(pool_size),
                    ..Default::default()
                };
                let response = model.solve_with_parameters(&params);
                // La solution principale peut déjà figurer dans le réservoir.
                let distinct: BTreeSet<_> = response.additional_solutions.iter()
                    .map(|s| s.values.as_slice())
                    .chain(std::iter::once(response.solution.as_slice()))
                    .filter_map(|values| values.get(..n))
                    .collect();
                count += distinct.len();
                let status = response.status();
                if status != CpSolverStatus::Optimal {
                    let status = if status == CpSolverStatus::Unknown && count > 0 {
                        CpSolverStatus::Feasible
                    } else {
                        status
                    };
                    return (count, status, start.elapsed().as_secs_f64() * 1_000.0);
                }
            }
        }
        // L'énumération globale est complète seulement si chaque partie est OPTIMAL.
        (count, CpSolverStatus::Optimal, start.elapsed().as_secs_f64() * 1_000.0)
    }

    let n = 8;
    let (count, status, ms) = count_nrooks_solutions_cpsat(n, Duration::from_secs(60));
    let factorial: usize = (1..=n).product();
    println!("Exemple : N-Tours (via CP-SAT)");
    println!("Statut global     : {}", format!("{status:?}").to_ascii_uppercase());
    println!("Solutions CP-SAT : {count}");
    println!("Solutions formule : {factorial} (= N!)");
    println!("Temps total : {ms:.1} ms");
    assert_eq!(status, CpSolverStatus::Optimal, "L'énumération doit être complète");
    assert_eq!(count, factorial);
};


Exemple : N-Tours (via CP-SAT)


Statut global     : OPTIMAL


Solutions CP-SAT : 40320


Solutions formule : 40320 (= N!)


Temps total : 1960.3 ms


### Exemple guide 1b : le problème des N-Fous

*Contribution d'un groupe d'étudiants (PR #20937), intégrée comme exemple guidé.*

**Enonce** : adaptez le problème pour placer le **maximum** de **fous** non attaquants sur un echiquier
$N \times N$. Contrairement aux reines et aux tours, les fous n'attaquent qu'en diagonale (pas en
ligne ni en colonne) : on peut donc en placer davantage que $N$.

**Consignes** :
1. Modelisez avec CP-SAT : une variable booléenne $b_{x,y}$ par case (1 si un fou l'occupe), et
   maximisez le nombre total de fous $k = \sum b_{x,y}$.
2. Posez les contraintes : au plus un fou par diagonale, c'est-à-dire pour chaque diagonale
   (constante en $x+y$ ou en $x-y$), la somme des $b_{x,y}$ sur ses cases vaut au plus 1.
3. Maximisez $k$ pour $N = 8$ : le maximum est $2N-2 = 14$ fous, disposés de **256** façons.

> **Note** : à la différence des N-Reines (exactement $N$ pièces, une par ligne), le problème des fous
> cherche le nombre *maximal* de pièces non attaquantes — un résultat classique valant $2N-2$. Dénombrer
> les placements d'*exactement* $N = 8$ fous (question différente) donnerait 22 522 960 configurations.

In [20]:
// EXEMPLE GUIDE 1b : N-Fous via CP-SAT (contribution etudiante, PR #20937)
{
    use cp_sat::builder::{BoolVar, CpModelBuilder, LinearExpr};
    use cp_sat::proto::{CpSolverStatus, SatParameters};
    use std::collections::BTreeSet;
    use std::time::Instant;

    type Placement = Vec<(usize, usize)>;

    fn build_bishops_model(n: usize) -> (CpModelBuilder, Vec<BoolVar>) {
        let mut model = CpModelBuilder::default();
        let b: Vec<_> = (0..n * n).map(|_| model.new_bool_var()).collect();
        let n = n as isize;
        for s in 0..2 * n - 1 { // diagonales x + y = s
            let cells = (0..n).filter_map(|x| {
                let y = s - x;
                (0..n).contains(&y).then(|| b[(x * n + y) as usize])
            });
            model.add_at_most_one(cells);
        }
        for d in (1 - n)..n { // diagonales x - y = d
            let cells = (0..n).filter_map(|x| {
                let y = x - d;
                (0..n).contains(&y).then(|| b[(x * n + y) as usize])
            });
            model.add_at_most_one(cells);
        }
        (model, b)
    }

    fn cardinality(b: &[BoolVar]) -> LinearExpr {
        b.iter().copied().map(LinearExpr::from).reduce(|sum, term| sum + term)
            .expect("L'échiquier contient au moins une case")
    }

    fn max_bishops_cpsat(n: usize) -> (Option<(usize, Placement)>, CpSolverStatus, f64) {
        // Première étape : maximiser le nombre de cases occupées.
        let (mut model, b) = build_bishops_model(n);
        model.maximize(cardinality(&b));
        let params = SatParameters { max_time_in_seconds: Some(10.0), ..Default::default() };
        let start = Instant::now();
        let response = model.solve_with_parameters(&params);
        let ms = start.elapsed().as_secs_f64() * 1_000.0;
        let status = response.status();
        let result = match status {
            CpSolverStatus::Optimal | CpSolverStatus::Feasible => {
                let placement: Placement = response.solution[..b.len()].iter().enumerate()
                    .filter_map(|(i, &occupied)| (occupied == 1).then_some((i / n, i % n)))
                    .collect();
                Some((placement.len(), placement))
            }
            _ => None,
        };
        (result, status, ms)
    }

    fn count_max_bishops_cpsat(n: usize, k: usize) -> (usize, CpSolverStatus, f64) {
        // Deuxième étape : fixer la cardinalité prouvée, sans objectif à optimiser.
        let (mut model, b) = build_bishops_model(n);
        model.add_eq(cardinality(&b), k as i64);
        let params = SatParameters {
            max_time_in_seconds: Some(30.0),
            num_search_workers: Some(1),
            enumerate_all_solutions: Some(true),
            fill_additional_solutions_in_response: Some(true),
            // Garder au-delà du compte attendu pour que l'assertion détecte aussi un excédent.
            solution_pool_size: Some(1 << (n + 1)),
            ..Default::default()
        };
        let start = Instant::now();
        let response = model.solve_with_parameters(&params);
        let ms = start.elapsed().as_secs_f64() * 1_000.0;
        let distinct: BTreeSet<_> = response.additional_solutions.iter().map(|s| s.values.as_slice())
            .chain(std::iter::once(response.solution.as_slice()))
            .filter_map(|values| values.get(..b.len()))
            .collect();
        (distinct.len(), response.status(), ms)
    }

    fn bishops_are_safe(placement: &[(usize, usize)]) -> bool {
        placement.iter().enumerate().all(|(i, &(x, y))| {
            placement[i + 1..].iter().all(|&(u, v)| x.abs_diff(u) != y.abs_diff(v))
        })
    }

    // --- Resolution pour N = 8 ---
    let n = 8;
    let (result, status, t_opt) = max_bishops_cpsat(n);
    assert_eq!(status, CpSolverStatus::Optimal);
    let (k_max, placement) = result.expect("Placement maximal attendu");
    let (ways, enum_status, t_enum) = count_max_bishops_cpsat(n, k_max);
    let safe = bishops_are_safe(&placement);
    println!("N = {n}");
    println!("  Maximum de fous     : {k_max} ({}, {t_opt:.1} ms)", format!("{status:?}").to_ascii_uppercase());
    println!("  Placements maximaux : {ways} ({}, {t_enum:.1} ms)", format!("{enum_status:?}").to_ascii_uppercase());
    println!("  Placement verifie hors solveur : {safe}");
    assert_eq!(enum_status, CpSolverStatus::Optimal);
    assert_eq!((k_max, ways), (14, 256));
    assert!(safe);

    // --- Generalisation et cas limite N = 1 ---
    println!("\n{:>3} {:>6} {:>6} {:>11} {:>6}", "N", "k_max", "2N-2", "placements", "2^N");
    for m in 1..=n {
        let bound = 2 * m - 2;
        let power = 1usize << m;
        let (k, count) = if m == n {
            (k_max, ways)
        } else {
            let (result, status, _) = max_bishops_cpsat(m);
            assert_eq!(status, CpSolverStatus::Optimal, "Optimisation pour N={m}");
            let (k, placement) = result.expect("Placement maximal attendu");
            assert!(bishops_are_safe(&placement));
            let (count, status, _) = count_max_bishops_cpsat(m, k);
            assert_eq!(status, CpSolverStatus::Optimal, "Énumération pour N={m}");
            (k, count)
        };
        println!("{m:>3} {k:>6} {bound:>6} {count:>11} {power:>6}");
        assert_eq!((k, count), if m == 1 { (1, 1) } else { (bound, power) });
    }
};


N = 8


  Maximum de fous     : 14 (OPTIMAL, 8.1 ms)


  Placements maximaux : 256 (OPTIMAL, 39.8 ms)


  Placement verifie hors solveur : true


  N  k_max   2N-2  placements    2^N


  1      1      0           1      2


... (3 outputs left out)

  5      8      8          32     32


  6     10     10          64     64


  7     12     12         128    128


  8     14     14         256    256


In [21]:
// EXEMPLE GUIDE 1b (verification independante) : denombrement exact par programmation dynamique
// CP-SAT ne peut pas raisonnablement enumerer les ~22,5 millions de placements
// de 8 fous cites dans la note de l'enonce. On les compte autrement : chaque
// diagonale montante (x + y = s) recoit au plus un fou ; l'etat memorise
// l'ensemble (bitmask) des diagonales descendantes (x - y) deja occupees.
{
    use std::collections::BTreeMap;

    #[derive(Clone, Copy)]
    struct State {
        ways: u64,
        previous: usize,
        bishop: Option<(usize, usize)>,
    }

    fn bishops_polynomial(n: usize) -> (Vec<u64>, Vec<(usize, usize)>) {
        assert!((1..=8).contains(&n));
        // masque -> coefficients par k : un masque de k bits ne contribue qu'au degré k.
        let initial = State { ways: 1, previous: 0, bishop: None };
        let mut layers = vec![BTreeMap::from([(0usize, initial)])];
        for s in 0..2 * n - 1 {
            let mut next = BTreeMap::<usize, State>::new();
            for (&mask, state) in layers.last().expect("Couche initiale présente") {
                let mut add = |new_mask, bishop| {
                    let entry = next.entry(new_mask).or_insert(State { ways: 0, previous: mask, bishop });
                    entry.ways += state.ways;
                };
                add(mask, None); // aucun fou sur cette diagonale
                for x in 0..n {
                    let Some(y) = s.checked_sub(x).filter(|&y| y < n) else { continue; };
                    let bit = 1usize << (x + n - 1 - y);
                    if mask & bit == 0 { // diagonale descendante libre
                        add(mask | bit, Some((x, y)));
                    }
                }
            }
            layers.push(next);
        }

        let mut polynomial = vec![0u64; 2 * n];
        let last = layers.last().expect("Dernière couche présente");
        for (&mask, state) in last {
            polynomial[mask.count_ones() as usize] += state.ways;
        }
        // Remonter un chemin de cardinalité maximale dans les couches de la DP.
        let mut mask = *last.keys().max_by_key(|&&mask| mask.count_ones()).expect("Placement vide présent");
        let mut placement = Vec::new();
        for layer in layers.iter().skip(1).rev() {
            let entry = layer[&mask];
            if let Some(bishop) = entry.bishop { placement.push(bishop); }
            mask = entry.previous;
        }
        (polynomial, placement)
    }

    let n = 8;
    let (p, placement) = bishops_polynomial(n);
    let k = p.iter().rposition(|&c| c > 0).expect("Placement vide présent");
    let count8 = p[8];
    println!("Programmation dynamique, N = {n} :");
    println!("  maximum de fous        : {k}");
    println!("  placements de {k} fous   : {}", p[k]);
    println!("  placements de 8 fous   : {} {:03} {:03}", count8 / 1_000_000, count8 / 1_000 % 1_000, count8 % 1_000);
    assert_eq!((k, p[k]), (14, 256)); // accord avec CP-SAT
    assert_eq!(count8, 22_522_960); // valeur citee dans l'enonce
    assert_eq!(placement.len(), k);
    assert!(placement.iter().enumerate().all(|(i, &(x, y))| {
        placement[i + 1..].iter().all(|&(u, v)| x.abs_diff(u) != y.abs_diff(v))
    }));

    // Visualisation de la solution optimale trouvee par CP-SAT dans le jumeau Python.
    // Ici, le plateau reste autonome : son placement est reconstruit par la DP.
    println!("\nPlacement maximal reconstruit par programmation dynamique :");
    for y in 0..n {
        let line: String = (0..n).map(|x| {
            if placement.contains(&(x, y)) { 'F' } else if (x + y) % 2 == 0 { '.' } else { '#' }
        }).collect();
        println!("{line}");
    }
};


Programmation dynamique, N = 8 :


  maximum de fous        : 14


  placements de 14 fous   : 256


  placements de 8 fous   : 22 522 960


Placement maximal reconstruit par programmation dynamique :


.#.#.#.F


... (3 outputs left out)

F#.#.#.F


F.#.#.#F


F#.#.#.F


#.#.#.#F


### Interprétation : exemple guide 1b (N-Fous)

**Sortie obtenue** : pour $N = 8$, CP-SAT trouve $k = 14 = 2N-2$ avec le statut `OPTIMAL` (le maximum est donc **prouvé**, pas seulement atteint), puis dénombre **256** placements maximaux. Un placement est revérifié hors solveur, et la programmation dynamique, qui est une méthode indépendante, retrouve 256 ainsi que les **22 522 960** placements d'exactement 8 fous cités dans la note.

| Choix de modélisation | Justification |
|---|---|
| Une booléenne $b_{x,y}$ par case | Les fous ne sont pas « un par colonne » (14 fous pour 8 colonnes) : la variable entière par colonne des N-Reines ne s'applique plus. |
| `add_at_most_one` par diagonale | C'est l'équivalent booléen de l'indice `add_all_different` sur $q_i + i$ et $q_i - i$ : chaque diagonale ($x+y$ ou $x-y$ constant) porte au plus une pièce. |
| Deux résolutions successives | On maximise d'abord $k$, puis on énumère avec la contrainte $\sum b = k$. L'énumération exhaustive de CP-SAT porte sur un problème de satisfaction, pas sur un problème d'optimisation. |

**Pourquoi $2N-2$ ?** Il y a $2N-1$ diagonales montantes, donc au plus $2N-1$ fous. Les deux diagonales extrêmes ($x+y = 0$ et $x+y = 2N-2$) se réduisent aux coins $(0,0)$ et $(N-1,N-1)$, qui sont sur la même diagonale descendante : un seul des deux peut être occupé, d'où $2N-2$. Cette borne est atteinte, par exemple avec $N$ fous sur la première colonne et $N-2$ sur la dernière (coins exclus).

**Cas limite $N = 1$** : le tableau donne 1 fou et 1 placement, alors que les formules donnent $2N-2 = 0$ et $2^N = 2$. L'argument des deux coins suppose deux coins distincts : les formules $2N-2$ et $2^N$ (dénombrement classique) ne valent que pour $N \geq 2$, ce que le tableau confirme pour $N = 2$ à $8$.

> **À retenir** : passer des reines aux fous change la **nature** des variables (entières, une par colonne, vers booléennes, une par case), et non seulement les contraintes. Le statut `OPTIMAL` et la vérification par une méthode indépendante (programmation dynamique) distinguent un résultat **prouvé** d'un résultat simplement observé.

### Exercice 1c : le problème des N-Rois

**Enonce** : placez le **maximum** de **rois** non attaquants sur un echiquier $N \times N$.
Les rois attaquent toutes les cases adjacentes, y compris en diagonale. Après les tours
(exemple guide 1) et les fous (exemple guide 1b), la contrainte change encore de nature :
elle ne porte plus sur des lignes ou des diagonales entières, mais sur des **voisinages locaux**.

**Consignes** :
1. Modelisez avec CP-SAT : une variable booléenne $b_{x,y}$ par case, et `add_at_most_one` sur
   chaque bloc $2 \times 2$ de cases adjacentes (un roi en $(x, y)$ interdit tout son
   voisinage, et le bloc est la façon la plus économique d'écrire cette interdiction).
2. Maximisez $k = \sum b_{x,y}$ pour $N = 8$ : le maximum attendu vaut $\lceil N/2 \rceil^2 = 16$.
3. Dénombrez les placements maximaux pour $N = 4$ et vérifiez l'un d'eux à la main sur papier
   (les compteurs CP-SAT de l'exemple guide 1b se transposent tels quels).

> **Indice** : les blocs $2 \times 2$ se chevauchent — une même case appartient à jusqu'à
> 4 blocs. Il y en a $(N-1)^2$ sur un echiquier $N \times N$.

In [22]:
// Exercice 1c : le problème des N-Rois
// TODO étudiant : construire le modele, maximiser, puis denombrer
{
    use cp_sat::builder::{BoolVar, CpModelBuilder};
    use cp_sat::proto::CpSolverStatus;

    type Placement = Vec<(usize, usize)>;
    type KingsModel = (CpModelBuilder, Vec<BoolVar>);

    fn build_kings_model(_n: usize) -> Option<KingsModel> {
        // TODO étudiant : (N-1)^2 blocs, add_at_most_one sur chacun.
        None
    }

    fn max_kings_cpsat(_n: usize, _time_limit_s: f64) -> Option<(usize, Placement, CpSolverStatus)> {
        // TODO étudiant : meme structure que max_bishops_cpsat.
        None
    }

    fn count_max_kings_cpsat(_n: usize, _k: usize, _time_limit_s: f64) -> Option<(usize, CpSolverStatus)> {
        // TODO étudiant : fixer k et énumérer.
        None
    }

    // --- Resolution pour N = 8 ---
    // TODO étudiant :
    // 1. k_max pour N = 8, a comparer a n.div_ceil(2).pow(2) (attendu : 16)
    // 2. une verification hors solveur, sur le modele de bishops_are_safe
    // 3. le denombrement pour N = 4, et un placement verifie a la main
    let _ = build_kings_model(8);
    let _ = max_kings_cpsat(8, 10.0);
    println!("Exercice a completer");
};


Exercice a completer


### Exemple guide 2 : seuil de croisement backtracking / min-conflicts

**Enonce** : trouvez experimentalement le plus petit $N$ pour lequel min-conflicts est systematiquement plus rapide que le backtracking FC+MRV.

1. Testez les deux approches pour $N = 8, 10, 12, 14, 16, 18, 20$
2. Moyennez sur 5 essais (min-conflicts est stochastique)
3. Tracez les courbes de temps et identifiez le point de croisement

In [23]:
// Exemple resolu : seuil de croisement backtracking / min-conflicts
{
    use std::time::Instant;

    // Comme dans le jumeau, la copie d'un domaine peut modifier l'ordre des lignes.
    #[derive(Clone, Copy, PartialEq, Eq)]
    enum Slot { Empty, Deleted, Row(usize) }
    #[derive(Clone)]
    struct Domain { slots: Vec<Slot>, len: usize, fill: usize }
    impl Domain {
        fn empty() -> Self { Self { slots: vec![Slot::Empty; 8], len: 0, fill: 0 } }
        fn rows(&self) -> impl Iterator<Item = usize> + '_ {
            self.slots.iter().filter_map(|s| match *s { Slot::Row(r) => Some(r), _ => None })
        }
        fn locate(&self, row: usize) -> Result<usize, usize> {
            let mask = self.slots.len() - 1;
            let (mut i, mut perturb, mut deleted) = (row & mask, row, None);
            loop {
                let end = if i + 9 <= mask { i + 9 } else { i };
                for j in i..=end {
                    match self.slots[j] {
                        Slot::Row(r) if r == row => return Ok(j),
                        Slot::Empty => return Err(deleted.unwrap_or(j)),
                        Slot::Deleted => deleted = Some(j),
                        _ => {}
                    }
                }
                perturb >>= 5;
                i = i.wrapping_mul(5).wrapping_add(1).wrapping_add(perturb) & mask;
            }
        }
        fn put(&mut self, row: usize) {
            if let Err(i) = self.locate(row) {
                if self.slots[i] == Slot::Empty { self.fill += 1; }
                self.slots[i] = Slot::Row(row);
                self.len += 1;
            }
        }
        fn rehash(&self, size: usize) -> Self {
            let mut out = Self { slots: vec![Slot::Empty; size], len: 0, fill: 0 };
            for row in self.rows() { out.put(row); }
            out
        }
        fn insert(&mut self, row: usize) {
            let fill = self.fill;
            self.put(row);
            if self.fill != fill && self.fill * 5 >= (self.slots.len() - 1) * 3 {
                *self = self.rehash((self.len * 4 + 1).next_power_of_two().max(8));
            }
        }
        fn remove(&mut self, row: usize) -> bool {
            if let Ok(i) = self.locate(row) {
                self.slots[i] = Slot::Deleted;
                self.len -= 1;
                true
            } else { false }
        }
        fn copy_domain(&self) -> Self {
            let size = if self.len < 5 { 8 } else { (2 * self.len + 1).next_power_of_two() };
            if size == self.slots.len() && self.fill == self.len { self.clone() }
            else { self.rehash(size) }
        }
    }

    fn backtracking_fc(n: usize) -> Option<(Vec<usize>, usize)> {
        fn visit(q: &mut [Option<usize>], d: &mut [Domain], nodes: &mut usize) -> bool {
            let n = q.len();
            let Some(col) = (0..n).filter(|&c| q[c].is_none())
                .min_by_key(|&c| (d[c].len, c)) else { return true; };
            let rows: Vec<_> = d[col].rows().collect();
            for row in rows {
                *nodes += 1;
                q[col] = Some(row);
                let old = d[col].copy_domain();
                d[col] = Domain::empty();
                let mut pruned = Vec::new();
                for c in 0..n {
                    if q[c].is_some() { continue; }
                    let gap = c.abs_diff(col);
                    for r in [Some(row), row.checked_sub(gap), row.checked_add(gap)].iter().copied().flatten() {
                        if r < n && d[c].remove(r) { pruned.push((c, r)); }
                    }
                }
                if !(0..n).any(|c| q[c].is_none() && d[c].len == 0) && visit(q, d, nodes) { return true; }
                for (c, r) in pruned { d[c].insert(r); }
                d[col] = old;
                q[col] = None;
            }
            false
        }
        let mut full = Domain::empty();
        for r in 0..n { full.insert(r); }
        let (mut q, mut d, mut nodes) = (vec![None; n], vec![full; n], 0);
        if visit(&mut q, &mut d, &mut nodes) {
            q.into_iter().collect::<Option<Vec<_>>>().map(|rows| (rows, nodes))
        } else { None }
    }

    // MT19937 : initialisation CPython pour les graines sur un mot de 32 bits.
    struct Mt19937 { state: [u32; 624], index: usize }
    impl Mt19937 {
        fn seeded(seed: u32) -> Self {
            let mut state = [0u32; 624];
            state[0] = 19_650_218;
            for i in 1..624 {
                let p = state[i - 1];
                state[i] = 1_812_433_253u32.wrapping_mul(p ^ (p >> 30)).wrapping_add(i as u32);
            }
            let mut i = 1;
            for _ in 0..624 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_664_525)).wrapping_add(seed);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            for _ in 0..623 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_566_083_941)).wrapping_sub(i as u32);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            state[0] = 0x8000_0000;
            Self { state, index: 624 }
        }
        fn next_u32(&mut self) -> u32 {
            if self.index == 624 {
                for i in 0..624 {
                    let y = (self.state[i] & 0x8000_0000) | (self.state[(i + 1) % 624] & 0x7fff_ffff);
                    self.state[i] = self.state[(i + 397) % 624] ^ (y >> 1) ^ if y & 1 == 0 { 0 } else { 0x9908_b0df };
                }
                self.index = 0;
            }
            let mut y = self.state[self.index];
            self.index += 1;
            y ^= y >> 11;
            y ^= (y << 7) & 0x9d2c_5680;
            y ^= (y << 15) & 0xefc6_0000;
            y ^ (y >> 18)
        }
        fn randbelow(&mut self, n: usize) -> usize {
            assert!(n > 0 && n <= u32::MAX as usize);
            let n = n as u32;
            loop {
                let r = self.next_u32() >> n.leading_zeros();
                if r < n { return r as usize; }
            }
        }
    }
    fn conflicts_at(q: &[usize], col: usize, row: usize) -> usize {
        q.iter().enumerate().filter(|&(c, &r)| c != col && (row == r || c.abs_diff(col) == row.abs_diff(r))).count()
    }
    fn total_conflicts(q: &[usize]) -> usize {
        q.iter().enumerate().map(|(c, &r)| {
            q.iter().enumerate().skip(c + 1).filter(|&(i, &y)| r == y || i - c == r.abs_diff(y)).count()
        }).sum()
    }
    fn min_conflicts(n: usize, rng: &mut Mt19937) -> (Option<Vec<usize>>, usize, Vec<usize>) {
        let mut q: Vec<_> = (0..n).collect();
        for i in (1..n).rev() { q.swap(i, rng.randbelow(i + 1)); }
        let bound = (5 * n).max(1_000);
        let mut history = vec![total_conflicts(&q)];
        for step in 0..bound {
            let conflicted: Vec<_> = (0..n).filter(|&c| conflicts_at(&q, c, q[c]) > 0).collect();
            if conflicted.is_empty() { return (Some(q), step, history); }
            let col = conflicted[rng.randbelow(conflicted.len())];
            let mut best = Vec::new();
            let mut minimum = usize::MAX;
            for row in 0..n {
                let score = conflicts_at(&q, col, row);
                if score < minimum { minimum = score; best.clear(); }
                if score == minimum { best.push(row); }
            }
            q[col] = best[rng.randbelow(best.len())];
            history.push(total_conflicts(&q));
        }
        (None, bound, history)
    }

    let mut results = Vec::new();
    for n in [8, 10, 12, 14, 16, 18, 20] {
        // Backtracking (deterministe)
        let start = Instant::now();
        let bt = backtracking_fc(n);
        let bt_ms = start.elapsed().as_secs_f64() * 1_000.0;
        let (q, nodes) = bt.expect("Solution FC+MRV attendue");
        assert_eq!(total_conflicts(&q), 0);

        // Min-conflicts (stochastique, moyenne sur 5 essais)
        let mut steps = [0; 5];
        let mut elapsed = 0.0;
        let mut successes = 0;
        for (trial, count) in steps.iter_mut().enumerate() {
            let mut rng = Mt19937::seeded((1_000 + 17 * trial + n) as u32);
            let start = Instant::now();
            let (solution, used, history) = min_conflicts(n, &mut rng);
            elapsed += start.elapsed().as_secs_f64() * 1_000.0;
            *count = used;
            if let Some(q) = solution {
                assert_eq!(history.last(), Some(&0));
                assert_eq!(total_conflicts(&q), 0);
                successes += 1;
            }
        }
        results.push((n, bt_ms, elapsed / steps.len() as f64, nodes, steps, successes));
    }

    // Trouver le seuil
    let first = results.iter().find(|row| row.5 == 5 && row.2 < row.1).map(|row| row.0);
    let stable = (0..results.len()).find(|&i| results[i..].iter().all(|row| row.5 == 5 && row.2 < row.1))
        .map(|i| results[i].0);
    // Plot : le tableau texte remplace les courbes de temps.
    println!("N | BT (ms) | MC moy. (ms) | Nœuds BT | Itérations MC par essai | Réussis");
    for (n, bt, mc, nodes, steps, successes) in results {
        println!("{n:>2} | {bt:>7.3} | {mc:>12.3} | {nodes:>8} | {steps:?} | {successes}/5");
    }
    match first {
        Some(n) => println!("Premier croisement : N = {n}"),
        None => println!("Pas de croisement sur cette plage"),
    }
    match stable {
        Some(n) => println!("Seuil stable sur la plage mesurée : N = {n}"),
        None => println!("Pas de seuil stable sur cette plage"),
    }
};


N | BT (ms) | MC moy. (ms) | Nœuds BT | Itérations MC par essai | Réussis


 8 |   0.116 |        0.394 |       75 | [16, 1000, 179, 1000, 63] | 3/5


10 |   0.067 |        0.232 |       35 | [87, 174, 178, 312, 22] | 5/5


12 |   0.258 |        0.259 |      153 | [60, 32, 328, 129, 17] | 5/5


14 |   0.172 |        0.243 |      140 | [566, 237, 64, 29, 135] | 5/5


16 |   0.059 |        0.185 |       39 | [86, 268, 22, 106, 50] | 5/5


18 |   0.083 |        0.125 |       55 | [63, 49, 71, 86, 27] | 5/5


20 |   0.189 |        0.375 |      145 | [40, 106, 71, 126, 82] | 5/5


Pas de croisement sur cette plage


Pas de seuil stable sur cette plage


### Exemple guide 2b : plage etendue jusqu'à N=50

*Contribution d'un groupe d'étudiants (PR #20937), intégrée comme exemple guidé.*

**Enonce** : relancez la comparaison sur une plage plus large :
$N \in \{20, 25, 30, 35, 40, 45, 50\}$.

**Consignes** :
1. Mesurez les temps de backtracking et min-conflicts pour chaque $N$
2. Affichez le graphe et identifiez le seuil
3. Attention : backtracking peut devenir très lent pour $N > 30$,
   ajoutez un `time_limit` raisonnable par taille (runtime *machine-dep*)

In [24]:
// EXEMPLE GUIDE 2b : plage etendue jusqu'a N=50 (contribution etudiante, PR #20937)
{
    use std::time::{Duration, Instant};

    // Reproduire l'ordre de visite des ensembles CPython de petits entiers.
    #[derive(Clone, Copy, PartialEq, Eq)]
    enum Slot { Empty, Deleted, Row(usize) }
    #[derive(Clone)]
    struct Domain { slots: Vec<Slot>, len: usize, fill: usize }
    impl Domain {
        fn empty() -> Self { Self { slots: vec![Slot::Empty; 8], len: 0, fill: 0 } }
        fn rows(&self) -> impl Iterator<Item = usize> + '_ {
            self.slots.iter().filter_map(|s| match *s { Slot::Row(r) => Some(r), _ => None })
        }
        fn locate(&self, row: usize) -> Result<usize, usize> {
            let mask = self.slots.len() - 1;
            let (mut i, mut perturb, mut deleted) = (row & mask, row, None);
            loop {
                let end = if i + 9 <= mask { i + 9 } else { i };
                for j in i..=end {
                    match self.slots[j] {
                        Slot::Row(r) if r == row => return Ok(j),
                        Slot::Empty => return Err(deleted.unwrap_or(j)),
                        Slot::Deleted => deleted = Some(j),
                        _ => {}
                    }
                }
                perturb >>= 5;
                i = i.wrapping_mul(5).wrapping_add(1).wrapping_add(perturb) & mask;
            }
        }
        fn put(&mut self, row: usize) {
            if let Err(i) = self.locate(row) {
                if self.slots[i] == Slot::Empty { self.fill += 1; }
                self.slots[i] = Slot::Row(row);
                self.len += 1;
            }
        }
        fn rehash(&self, size: usize) -> Self {
            let mut out = Self { slots: vec![Slot::Empty; size], len: 0, fill: 0 };
            for row in self.rows() { out.put(row); }
            out
        }
        fn insert(&mut self, row: usize) {
            let fill = self.fill;
            self.put(row);
            if self.fill != fill && self.fill * 5 >= (self.slots.len() - 1) * 3 {
                *self = self.rehash((self.len * 4 + 1).next_power_of_two().max(8));
            }
        }
        fn remove(&mut self, row: usize) -> bool {
            if let Ok(i) = self.locate(row) {
                self.slots[i] = Slot::Deleted;
                self.len -= 1;
                true
            } else { false }
        }
        fn copy_domain(&self) -> Self {
            let size = if self.len < 5 { 8 } else { (2 * self.len + 1).next_power_of_two() };
            if size == self.slots.len() && self.fill == self.len { self.clone() }
            else { self.rehash(size) }
        }
    }

    #[derive(Clone, Copy, Debug, PartialEq, Eq)]
    enum BtStatus { Ok, Timeout, Echec }
    impl BtStatus {
        fn label(self) -> &'static str {
            match self { Self::Ok => "OK", Self::Timeout => "TIMEOUT", Self::Echec => "ECHEC" }
        }
    }
    fn run_fc(n: usize, limit: Option<Duration>) -> (Option<Vec<usize>>, usize, BtStatus) {
        fn visit(q: &mut [Option<usize>], d: &mut [Domain], nodes: &mut usize, deadline: Option<Instant>) -> BtStatus {
            let n = q.len();
            let Some(col) = (0..n).filter(|&c| q[c].is_none())
                .min_by_key(|&c| (d[c].len, c)) else { return BtStatus::Ok; };
            let rows: Vec<_> = d[col].rows().collect();
            for row in rows {
                *nodes += 1;
                // Vérifier le délai au même point que dans la version Python.
                if let Some(end) = deadline {
                    if Instant::now() > end { return BtStatus::Timeout; }
                }
                q[col] = Some(row);
                let old = d[col].copy_domain();
                d[col] = Domain::empty();
                let mut pruned = Vec::new();
                for c in 0..n {
                    if q[c].is_some() { continue; }
                    let gap = c.abs_diff(col);
                    for r in [Some(row), row.checked_sub(gap), row.checked_add(gap)].iter().copied().flatten() {
                        if r < n && d[c].remove(r) { pruned.push((c, r)); }
                    }
                }
                if !(0..n).any(|c| q[c].is_none() && d[c].len == 0) {
                    match visit(q, d, nodes, deadline) {
                        BtStatus::Ok => return BtStatus::Ok,
                        BtStatus::Timeout => return BtStatus::Timeout,
                        BtStatus::Echec => {}
                    }
                }
                for (c, r) in pruned { d[c].insert(r); }
                d[col] = old;
                q[col] = None;
            }
            BtStatus::Echec
        }
        let mut full = Domain::empty();
        for r in 0..n { full.insert(r); }
        let (mut q, mut d, mut nodes) = (vec![None; n], vec![full; n], 0);
        let deadline = limit.map(|duration| Instant::now() + duration);
        let status = visit(&mut q, &mut d, &mut nodes, deadline);
        let solution = if status == BtStatus::Ok { q.into_iter().collect::<Option<Vec<_>>>() } else { None };
        (solution, nodes, status)
    }
    fn backtracking_fc(n: usize) -> Option<(Vec<usize>, usize)> {
        let (q, nodes, _) = run_fc(n, None);
        q.map(|rows| (rows, nodes))
    }
    fn backtracking_fc_timed(n: usize, limit: Duration) -> (Option<Vec<usize>>, usize, BtStatus) {
        run_fc(n, Some(limit))
    }

    // MT19937 : initialisation CPython pour les graines sur un mot de 32 bits.
    struct Mt19937 { state: [u32; 624], index: usize }
    impl Mt19937 {
        fn seeded(seed: u32) -> Self {
            let mut state = [0u32; 624];
            state[0] = 19_650_218;
            for i in 1..624 {
                let p = state[i - 1];
                state[i] = 1_812_433_253u32.wrapping_mul(p ^ (p >> 30)).wrapping_add(i as u32);
            }
            let mut i = 1;
            for _ in 0..624 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_664_525)).wrapping_add(seed);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            for _ in 0..623 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_566_083_941)).wrapping_sub(i as u32);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            state[0] = 0x8000_0000;
            Self { state, index: 624 }
        }
        fn next_u32(&mut self) -> u32 {
            if self.index == 624 {
                for i in 0..624 {
                    let y = (self.state[i] & 0x8000_0000) | (self.state[(i + 1) % 624] & 0x7fff_ffff);
                    self.state[i] = self.state[(i + 397) % 624] ^ (y >> 1) ^ if y & 1 == 0 { 0 } else { 0x9908_b0df };
                }
                self.index = 0;
            }
            let mut y = self.state[self.index];
            self.index += 1;
            y ^= y >> 11;
            y ^= (y << 7) & 0x9d2c_5680;
            y ^= (y << 15) & 0xefc6_0000;
            y ^ (y >> 18)
        }
        fn randbelow(&mut self, n: usize) -> usize {
            assert!(n > 0 && n <= u32::MAX as usize);
            let n = n as u32;
            loop {
                let r = self.next_u32() >> n.leading_zeros();
                if r < n { return r as usize; }
            }
        }
    }
    fn conflicts_at(q: &[usize], col: usize, row: usize) -> usize {
        q.iter().enumerate().filter(|&(c, &r)| c != col && (row == r || c.abs_diff(col) == row.abs_diff(r))).count()
    }
    fn total_conflicts(q: &[usize]) -> usize {
        q.iter().enumerate().map(|(c, &r)| {
            q.iter().enumerate().skip(c + 1).filter(|&(i, &y)| r == y || i - c == r.abs_diff(y)).count()
        }).sum()
    }
    fn min_conflicts(n: usize, rng: &mut Mt19937) -> (Option<Vec<usize>>, usize, Vec<usize>) {
        let mut q: Vec<_> = (0..n).collect();
        for i in (1..n).rev() { q.swap(i, rng.randbelow(i + 1)); }
        let bound = (5 * n).max(1_000);
        let mut history = vec![total_conflicts(&q)];
        for step in 0..bound {
            let conflicted: Vec<_> = (0..n).filter(|&c| conflicts_at(&q, c, q[c]) > 0).collect();
            if conflicted.is_empty() { return (Some(q), step, history); }
            let col = conflicted[rng.randbelow(conflicted.len())];
            let mut best = Vec::new();
            let mut minimum = usize::MAX;
            for row in 0..n {
                let score = conflicts_at(&q, col, row);
                if score < minimum { minimum = score; best.clear(); }
                if score == minimum { best.push(row); }
            }
            q[col] = best[rng.randbelow(best.len())];
            history.push(total_conflicts(&q));
        }
        (None, bound, history)
    }

    // Controle de coherence : la version chronometree reproduit backtracking_fc
    for n in [8, 12, 16, 20] {
        let expected = backtracking_fc(n).expect("Solution de référence attendue");
        let (q, nodes, status) = backtracking_fc_timed(n, Duration::from_secs(10));
        assert_eq!(status, BtStatus::Ok);
        assert_eq!((q.expect("Solution chronométrée attendue"), nodes), expected);
    }
    println!("Version chronometree identique a backtracking_fc sur N = 8, 12, 16, 20");
    let limit = Duration::from_secs(5); // limite par taille pour le backtracking (machine-dep)
    let cases = [(20, 145), (25, 219), (30, 181), (35, 271), (40, 132), (45, 589), (50, 2_074)];
    let mut wins = Vec::new();
    println!("N | BT FC+MRV (ms) | Nœuds | Statut | MC moy. (ms) | MC iter moy. | Réussis | Itérations par essai");
    for (n, expected_nodes) in cases {
        // Backtracking FC+MRV (deterministe, une seule mesure)
        let start = Instant::now();
        let (q, nodes, status) = backtracking_fc_timed(n, limit);
        let bt_ms = start.elapsed().as_secs_f64() * 1_000.0;
        if let Some(q) = q { assert_eq!(total_conflicts(&q), 0); }

        // Min-conflicts (stochastique) : moyenne sur 5 essais, memes graines que l'exemple
        let mut steps = [0; 5];
        let mut mc_ms = 0.0;
        let mut successes = 0;
        for (trial, count) in steps.iter_mut().enumerate() {
            let mut rng = Mt19937::seeded((1_000 + 17 * trial + n) as u32);
            let start = Instant::now();
            let (q, used, history) = min_conflicts(n, &mut rng);
            mc_ms += start.elapsed().as_secs_f64() * 1_000.0;
            *count = used;
            if let Some(q) = q {
                assert_eq!(total_conflicts(&q), 0);
                assert_eq!(history.last(), Some(&0));
                successes += 1;
            }
        }
        mc_ms /= steps.len() as f64;
        let mean_steps = steps.iter().sum::<usize>() as f64 / steps.len() as f64;
        println!("{n:>2} | {bt_ms:>14.3} | {nodes:>5} | {:>7} | {mc_ms:>12.3} | {mean_steps:>12.1} | {successes}/5 | {steps:?}", status.label());
        if status == BtStatus::Ok { assert_eq!(nodes, expected_nodes, "Nœuds pour N={n}"); }
        assert_eq!(successes, 5, "Essais Min-Conflicts réussis pour N={n}");
        wins.push((n, status == BtStatus::Timeout || (status == BtStatus::Ok && mc_ms < bt_ms)));
    }

    // Seuil : plus petit N a partir duquel min-conflicts gagne pour TOUS les N suivants
    // (un TIMEOUT du backtracking compte comme une victoire de min-conflicts)
    println!("Min-conflicts plus rapide : {wins:?}");
    let threshold = (0..wins.len()).find(|&i| wins[i..].iter().all(|&(_, win)| win));
    match threshold {
        Some(i) => println!("Seuil (min-conflicts systematiquement plus rapide) : N = {}", wins[i].0),
        None => println!("Pas de seuil stable sur cette plage"),
    }
};


Version chronometree identique a backtracking_fc sur N = 8, 12, 16, 20


N | BT FC+MRV (ms) | Nœuds | Statut | MC moy. (ms) | MC iter moy. | Réussis | Itérations par essai


20 |          0.220 |   145 |      OK |        0.233 |         85.0 | 5/5 | [40, 106, 71, 126, 82]


25 |          0.333 |   219 |      OK |        0.349 |         84.2 | 5/5 | [150, 25, 125, 44, 77]


30 |          0.286 |   181 |      OK |        0.480 |         82.4 | 5/5 | [59, 96, 40, 114, 103]


35 |          0.398 |   271 |      OK |        0.953 |        125.4 | 5/5 | [94, 45, 106, 190, 192]


40 |          0.357 |   132 |      OK |        1.406 |         93.8 | 5/5 | [115, 112, 87, 112, 43]


45 |          1.325 |   589 |      OK |        1.187 |        103.0 | 5/5 | [119, 112, 112, 125, 47]


50 |          3.222 |  2074 |      OK |        1.483 |         87.8 | 5/5 | [96, 78, 66, 134, 65]


Min-conflicts plus rapide : [(20, false), (25, false), (30, false), (35, false), (40, false), (45, true), (50, true)]


Seuil (min-conflicts systematiquement plus rapide) : N = 45


#### Complément : prolonger l'expérience au-delà de $N = 50$

*Contribution d'un groupe d'étudiants (PR #20937), intégrée comme exemple guidé.*

Sur la plage demandée, le backtracking FC+MRV reste plus rapide jusqu'à $N = 40$ ; à $N = 45$ et $N = 50$, min-conflicts passe devant dans l'exécution committée — un basculement qui **dépend de la machine** : l'écart entre les deux approches vaut **0,138 ms** à $N = 45$ (1,325 contre 1,187 ms) mais **1,739 ms** à $N = 50$ (3,222 contre 1,483 ms) ; seul le premier tient dans quelques dixièmes de milliseconde. Pour savoir **si** un croisement stable existe et **où** il se situe, on prolonge l'expérience jusqu'à $N = 200$ (limite de 5 s pour le backtracking, 3 essais pour min-conflicts). On relève aussi des mesures indépendantes de la machine : les nœuds explorés par le backtracking et les itérations de min-conflicts.

In [25]:
// EXEMPLE GUIDE 2b (complement) : au-dela de N = 50, ou se trouve le croisement ?
// Mesures independantes de la machine : noeuds explores (BT) et iterations (MC).
// Un compte de nœuds interrompu, contrairement à un compte achevé, dépend du délai.
{
    use std::time::{Duration, Instant};

    #[derive(Clone, Copy, PartialEq, Eq)]
    enum Slot { Empty, Deleted, Row(usize) }
    #[derive(Clone)]
    struct Domain { slots: Vec<Slot>, len: usize, fill: usize }
    impl Domain {
        fn empty() -> Self { Self { slots: vec![Slot::Empty; 8], len: 0, fill: 0 } }
        fn rows(&self) -> impl Iterator<Item = usize> + '_ {
            self.slots.iter().filter_map(|s| match *s { Slot::Row(r) => Some(r), _ => None })
        }
        fn locate(&self, row: usize) -> Result<usize, usize> {
            let mask = self.slots.len() - 1;
            let (mut i, mut perturb, mut deleted) = (row & mask, row, None);
            loop {
                let end = if i + 9 <= mask { i + 9 } else { i };
                for j in i..=end {
                    match self.slots[j] {
                        Slot::Row(r) if r == row => return Ok(j),
                        Slot::Empty => return Err(deleted.unwrap_or(j)),
                        Slot::Deleted => deleted = Some(j),
                        _ => {}
                    }
                }
                perturb >>= 5;
                i = i.wrapping_mul(5).wrapping_add(1).wrapping_add(perturb) & mask;
            }
        }
        fn put(&mut self, row: usize) {
            if let Err(i) = self.locate(row) {
                if self.slots[i] == Slot::Empty { self.fill += 1; }
                self.slots[i] = Slot::Row(row);
                self.len += 1;
            }
        }
        fn rehash(&self, size: usize) -> Self {
            let mut out = Self { slots: vec![Slot::Empty; size], len: 0, fill: 0 };
            for row in self.rows() { out.put(row); }
            out
        }
        fn insert(&mut self, row: usize) {
            let fill = self.fill;
            self.put(row);
            if self.fill != fill && self.fill * 5 >= (self.slots.len() - 1) * 3 {
                *self = self.rehash((self.len * 4 + 1).next_power_of_two().max(8));
            }
        }
        fn remove(&mut self, row: usize) -> bool {
            if let Ok(i) = self.locate(row) {
                self.slots[i] = Slot::Deleted;
                self.len -= 1;
                true
            } else { false }
        }
        fn copy_domain(&self) -> Self {
            let size = if self.len < 5 { 8 } else { (2 * self.len + 1).next_power_of_two() };
            if size == self.slots.len() && self.fill == self.len { self.clone() }
            else { self.rehash(size) }
        }
    }

    #[derive(Clone, Copy, PartialEq, Eq)]
    enum BtStatus { Ok, Timeout, Echec }
    impl BtStatus {
        fn label(self) -> &'static str {
            match self { Self::Ok => "OK", Self::Timeout => "TIMEOUT", Self::Echec => "ECHEC" }
        }
    }
    fn backtracking_fc_timed(n: usize, limit: Duration) -> (Option<Vec<usize>>, usize, BtStatus) {
        fn visit(q: &mut [Option<usize>], d: &mut [Domain], nodes: &mut usize, deadline: Instant) -> BtStatus {
            let n = q.len();
            let Some(col) = (0..n).filter(|&c| q[c].is_none())
                .min_by_key(|&c| (d[c].len, c)) else { return BtStatus::Ok; };
            let rows: Vec<_> = d[col].rows().collect();
            for row in rows {
                *nodes += 1;
                if Instant::now() > deadline { return BtStatus::Timeout; }
                q[col] = Some(row);
                let old = d[col].copy_domain();
                d[col] = Domain::empty();
                let mut pruned = Vec::new();
                for c in 0..n {
                    if q[c].is_some() { continue; }
                    let gap = c.abs_diff(col);
                    for r in [Some(row), row.checked_sub(gap), row.checked_add(gap)].iter().copied().flatten() {
                        if r < n && d[c].remove(r) { pruned.push((c, r)); }
                    }
                }
                if !(0..n).any(|c| q[c].is_none() && d[c].len == 0) {
                    match visit(q, d, nodes, deadline) {
                        BtStatus::Ok => return BtStatus::Ok,
                        BtStatus::Timeout => return BtStatus::Timeout,
                        BtStatus::Echec => {}
                    }
                }
                for (c, r) in pruned { d[c].insert(r); }
                d[col] = old;
                q[col] = None;
            }
            BtStatus::Echec
        }
        let mut full = Domain::empty();
        for r in 0..n { full.insert(r); }
        let (mut q, mut d, mut nodes) = (vec![None; n], vec![full; n], 0);
        let deadline = Instant::now() + limit;
        let status = visit(&mut q, &mut d, &mut nodes, deadline);
        let solution = if status == BtStatus::Ok { q.into_iter().collect::<Option<Vec<_>>>() } else { None };
        (solution, nodes, status)
    }

    // MT19937 avec la même initialisation et le même tirage par rejet qu'en c51.
    struct Mt19937 { state: [u32; 624], index: usize }
    impl Mt19937 {
        fn seeded(seed: u32) -> Self {
            let mut state = [0u32; 624];
            state[0] = 19_650_218;
            for i in 1..624 {
                let p = state[i - 1];
                state[i] = 1_812_433_253u32.wrapping_mul(p ^ (p >> 30)).wrapping_add(i as u32);
            }
            let mut i = 1;
            for _ in 0..624 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_664_525)).wrapping_add(seed);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            for _ in 0..623 {
                let p = state[i - 1];
                state[i] = (state[i] ^ (p ^ (p >> 30)).wrapping_mul(1_566_083_941)).wrapping_sub(i as u32);
                i += 1;
                if i == 624 { state[0] = state[623]; i = 1; }
            }
            state[0] = 0x8000_0000;
            Self { state, index: 624 }
        }
        fn next_u32(&mut self) -> u32 {
            if self.index == 624 {
                for i in 0..624 {
                    let y = (self.state[i] & 0x8000_0000) | (self.state[(i + 1) % 624] & 0x7fff_ffff);
                    self.state[i] = self.state[(i + 397) % 624] ^ (y >> 1) ^ if y & 1 == 0 { 0 } else { 0x9908_b0df };
                }
                self.index = 0;
            }
            let mut y = self.state[self.index];
            self.index += 1;
            y ^= y >> 11;
            y ^= (y << 7) & 0x9d2c_5680;
            y ^= (y << 15) & 0xefc6_0000;
            y ^ (y >> 18)
        }
        fn randbelow(&mut self, n: usize) -> usize {
            assert!(n > 0 && n <= u32::MAX as usize);
            let n = n as u32;
            loop {
                let r = self.next_u32() >> n.leading_zeros();
                if r < n { return r as usize; }
            }
        }
    }
    fn conflicts_at(q: &[usize], col: usize, row: usize) -> usize {
        q.iter().enumerate().filter(|&(c, &r)| c != col && (row == r || c.abs_diff(col) == row.abs_diff(r))).count()
    }
    fn total_conflicts(q: &[usize]) -> usize {
        q.iter().enumerate().map(|(c, &r)| {
            q.iter().enumerate().skip(c + 1).filter(|&(i, &y)| r == y || i - c == r.abs_diff(y)).count()
        }).sum()
    }
    fn min_conflicts(n: usize, rng: &mut Mt19937) -> (Option<Vec<usize>>, usize, Vec<usize>) {
        let mut q: Vec<_> = (0..n).collect();
        for i in (1..n).rev() { q.swap(i, rng.randbelow(i + 1)); }
        let bound = (5 * n).max(1_000);
        let mut history = vec![total_conflicts(&q)];
        for step in 0..bound {
            let conflicted: Vec<_> = (0..n).filter(|&c| conflicts_at(&q, c, q[c]) > 0).collect();
            if conflicted.is_empty() { return (Some(q), step, history); }
            let col = conflicted[rng.randbelow(conflicted.len())];
            let mut best = Vec::new();
            let mut minimum = usize::MAX;
            for row in 0..n {
                let score = conflicts_at(&q, col, row);
                if score < minimum { minimum = score; best.clear(); }
                if score == minimum { best.push(row); }
            }
            q[col] = best[rng.randbelow(best.len())];
            history.push(total_conflicts(&q));
        }
        (None, bound, history)
    }

    // Témoins Python : certains comptes BT ont été relevés lors d'une interruption.
    let cases = [
        (60, 1_716, true, 943), (80, 211, true, 1_353), (100, 198, true, 1_780),
        (110, 123_519, false, 1_307), (120, 115_484, false, 1_487), (150, 120_783, false, 1_727),
        (160, 367, true, 2_023), (200, 108_572, false, 1_993),
    ];
    let limit = Duration::from_secs(5);
    let mut timeouts = 0;
    let mut ratios = Vec::new();
    println!("N | BT (ms) | Nœuds BT | Statut | MC moy. (ms) | MC iter moy. | Gagnant | Nœuds Python");
    for (n, python_nodes, python_finished, expected_tenths) in cases {
        let start = Instant::now();
        let (q, nodes, status) = backtracking_fc_timed(n, limit);
        let bt_ms = start.elapsed().as_secs_f64() * 1_000.0;
        if let Some(q) = q { assert_eq!(total_conflicts(&q), 0); }
        if status == BtStatus::Timeout { timeouts += 1; }

        let mut sum_steps = 0;
        let mut sum_ms = 0.0;
        for trial in 0..3 {
            let mut rng = Mt19937::seeded((1_000 + 17 * trial + n) as u32);
            let start = Instant::now();
            let (q, used, history) = min_conflicts(n, &mut rng);
            sum_ms += start.elapsed().as_secs_f64() * 1_000.0;
            sum_steps += used;
            let q = q.expect("Min-Conflicts doit résoudre cet essai");
            assert_eq!(history.last(), Some(&0));
            assert_eq!(total_conflicts(&q), 0);
        }
        let mc_ms = sum_ms / 3.0;
        let mc_steps = sum_steps as f64 / 3.0;
        let winner = if status != BtStatus::Ok || mc_ms < bt_ms { "MC" } else { "BT" };
        let python_note = if python_finished { "achevé" } else { "interrompu" };
        println!("{n:>3} | {bt_ms:>8.3} | {nodes:>8} | {:>7} | {mc_ms:>12.3} | {mc_steps:>12.1} | {winner:>7} | {python_nodes} ({python_note})", status.label());
        assert_eq!((mc_steps * 10.0).round() as usize, expected_tenths, "Moyenne des itérations pour N={n}");
        if python_finished && status == BtStatus::Ok {
            assert_eq!(nodes, python_nodes, "Nœuds des recherches achevées pour N={n}");
        }
        ratios.push(format!("{:.2}", mc_steps / n as f64));
    }
    println!("\nBacktracking : {timeouts}/{} tailles en TIMEOUT ({} s)", cases.len(), limit.as_secs());
    println!("Min-conflicts : toutes les tailles resolues, iterations / N = [{}]", ratios.join(", "));
    println!("Les nœuds à l'interruption, les TIMEOUT et les gagnants sont remesurés, sans alignement sur Python.");
};


N | BT (ms) | Nœuds BT | Statut | MC moy. (ms) | MC iter moy. | Gagnant | Nœuds Python


 60 |    2.897 |     1716 |      OK |        1.850 |         94.3 |      MC | 1716 (achevé)


 80 |    1.067 |      211 |      OK |        4.890 |        135.3 |      BT | 211 (achevé)


100 |    1.432 |      198 |      OK |        8.902 |        178.0 |      BT | 198 (achevé)


110 | 5000.135 |  2126120 | TIMEOUT |        8.835 |        130.7 |      MC | 123519 (interrompu)


120 | 5000.140 |  1712574 | TIMEOUT |       10.080 |        148.7 |      MC | 115484 (interrompu)


... (2 outputs left out)

200 | 1795.687 |   654130 |      OK |       42.570 |        199.3 |      MC | 108572 (interrompu)


Backtracking : 2/8 tailles en TIMEOUT (5 s)


Min-conflicts : toutes les tailles resolues, iterations / N = [1.57, 1.69, 1.78, 1.19, 1.24, 1.15, 1.26, 1.00]


Les nœuds à l'interruption, les TIMEOUT et les gagnants sont remesurés, sans alignement sur Python.


### Interprétation : exemple guide 2b (plage étendue)

**Sortie obtenue** : sur $N \in \{20, \ldots, 50\}$, le backtracking FC+MRV est **plus rapide de $N = 20$ à $N = 40$** (132 à 2 074 nœuds, aucun timeout) ; à $N = 45$ et $N = 50$, min-conflicts passe devant ; l'écart vaut 0,138 ms au premier (1,325 contre 1,187 ms) et 1,739 ms au second (3,222 contre 1,483 ms), et **dépend de la machine**. Min-conflicts résout tous les essais (5/5) en 82 à 125 itérations en moyenne. **Il n'y a donc pas de seuil stable sur cette plage.** La version chronométrée est d'abord contrôlée : elle reproduit exactement les solutions et les nœuds de `backtracking_fc`.

| Constat | Explication |
|---|---|
| L'avertissement « le backtracking devient très lent pour $N > 30$ » ne se vérifie pas ici | Il vaut pour le backtracking **simple** (limité à $N \leq 20$ dans le benchmark de la section 6). Avec MRV et forward checking, la plupart des tailles se résolvent en quelques centaines de nœuds. |
| Min-conflicts perd sur la majeure partie de la plage malgré un faible nombre d'itérations | Dans cette implémentation, chaque itération coûte $O(N^2)$ : `conflicts_at` est recalculé pour chaque colonne, et `total_conflicts` est appelé à chaque pas pour l'historique. Un nœud de FC+MRV coûte $O(N)$. |

**Complément ($N$ jusqu'à 200)** : le comportement du backtracking devient **irrégulier**. Il dépasse la limite de 5 s pour 2 tailles sur 8 ($N = 110$ et $N = 120$, après plus de 1,7 million de nœuds chacune), alors que $N = 80$, $N = 100$ et $N = 160$ ne demandent que 198 à 367 nœuds, et que $N = 150$ et $N = 200$ aboutissent en moins de 3 s après 650 000 à 1,35 million de nœuds. La dernière colonne du tableau, « Nœuds Python », rappelle les nœuds comptés par le jumeau Python : pour les tailles interrompues, c'est un compte **partiel** au moment du timeout, non comparable aux nœuds Rust, qui sont ceux d'une exploration beaucoup plus profonde dans la même limite. Min-conflicts résout toutes les tailles en 1,0 à 1,8 fois $N$ itérations : une croissance **linéaire** du nombre d'itérations, conforme au résultat de Minton et al. (1992).

**Réponse à la question** : avec ces implémentations, il n'existe pas de $N$ à partir duquel min-conflicts serait *systématiquement* plus rapide. Dans ce jumeau, il gagne pour la première fois à $N = 45$ sur la plage courte, puis de façon entrelacée au-delà ($N$ = 60, 110, 120, 150 et 200 — dont deux par timeout du backtracking), le backtracking gardant l'avantage à $N$ = 80, 100 et 160. Le vrai critère n'est pas un seuil de vitesse mais la **robustesse** : au-delà de $N \approx 100$, le backtracking devient imprévisible, alors que min-conflicts reste prévisible.

> **Remarques** :
> - Les temps dépendent de la machine. En revanche, les nœuds, les itérations et les solutions sont déterministes (graines fixées), de même que la liste des tailles en timeout pour une limite donnée, à la vitesse de la machine près.
> - Cette variabilité extrême d'une instance à l'autre est le phénomène de **queue lourde** des recherches arborescentes (Gomes, Selman, Crato & Kautz, 2000, *Journal of Automated Reasoning* 24). On le traite classiquement par randomisation et redémarrages.
> - Une version incrémentale de min-conflicts (compteurs par ligne et par diagonale, $O(N)$ par itération) déplacerait vraisemblablement le croisement vers des $N$ plus petits. C'est une hypothèse, non mesurée ici.

### Exercice 2c : min-conflicts incrémental

**Enonce** : l'interprétation de l'exemple guide 2b se termine sur une hypothèse non
mesurée : une version **incrémentale** de min-conflicts, où chaque itération coûterait
$O(N)$ au lieu du $O(N^2)$ de l'implémentation du notebook, déplacerait le croisement
vers des $N$ plus petits. Construisez cette version et mesurez.

**Consignes** :
1. Implémentez `min_conflicts_fast(n)` : maintenez des compteurs par ligne et par
   diagonale (montante et descendante), mis à jour à chaque déplacement, pour évaluer
   en $O(1)$ le conflit d'une reine et le coût d'un déplacement, et choisissez la reine
   et sa colonne en $O(N)$ par itération. L'algorithme (choix aléatoires compris) doit
   rester celui de `min_conflicts`.
2. Contrôlez la version rapide : mêmes graines, mêmes solutions valides et même nombre
   d'itérations que `min_conflicts` sur $N = 20, 30, 40, 50$ — seul le coût par itération
   doit changer, pas la trajectoire.
3. Mesurez le nouveau croisement contre le backtracking FC+MRV (`backtracking_fc_timed`,
   exemple guide 2b) sur $N = 20$ à $50$ : à partir de quel $N$ la version incrémentale
   gagne-t-elle systématiquement, là où la version $O(N^2)$ perdait à chaque taille ?

> **Indice** : `conflicts_at` recompte toute la ligne et les diagonales à chaque
> appel ; un tableau `row_count[r]` et deux tableaux de diagonales suffisent pour
> répondre en $O(1)$ à « combien de reines attaquent la case $(c, r)$ ? ».

In [26]:
// Exercice 2c : min-conflicts incremental
// TODO étudiant : meme trajectoire que min_conflicts, cout par iteration en O(N)
{
    type SearchResult = (Option<Vec<usize>>, usize, Vec<usize>);

    fn min_conflicts_fast(_n: usize, _max_steps: Option<usize>, _seed: u32) -> Option<SearchResult> {
        // TODO étudiant : initialiser les compteurs puis boucler.
        None
    }

    // --- Controle : memes graines, memes trajectoires que min_conflicts ---
    // TODO étudiant : pour n_check dans [20, 30, 40, 50], verifier que
    // Mt19937::seeded(1000 + 17 * trial + n_check) donne la meme solution et les
    // memes steps que min_conflicts, et que total_conflicts(&sol) == 0.

    // --- Mesure : croisement contre backtracking_fc_timed sur N = 20..50 ---
    // TODO étudiant : reprendre le protocole de l'exemple guide 2b et identifier
    // le plus petit N a partir duquel min_conflicts_fast gagne systematiquement.
    let _ = min_conflicts_fast(20, None, 1020);
    println!("Exercice a completer");
};


Exercice a completer


### Exemple guide 3 : brisure de symetrie avec backtracking

**Enonce** : le problème des N-Reines admet des symetries (rotation de 90/180/270 degrés, reflexions horizontale et verticale). On peut reduire l'espace de recherche en ajoutant des **contraintes de brisure de symetrie**.

1. Ajoutez la contrainte `queens[0] < queens[N-1]` (elimine la reflexion horizontale)
2. Ajoutez `queens[0] < N / 2` (elimine les rotations de 180 degrés)
3. Comptez les solutions "fondamentales" (uniques sous symetrie) pour $N = 8$
4. Verifiez : il devrait y en avoir **12** solutions fondamentales

In [27]:
// Exemple resolu : brisure de symetrie
{
    use std::collections::BTreeSet;
    use std::time::Instant;

    fn enumerate_nqueens_solutions_backtracking(n: usize) -> Vec<Vec<usize>> {
        fn visit(q: &mut [usize], col: usize, solutions: &mut Vec<Vec<usize>>) {
            if col == q.len() {
                solutions.push(q.to_vec());
                return;
            }
            for row in 0..q.len() {
                if q[..col].iter().enumerate().all(|(c, &r)| r != row && col - c != r.abs_diff(row)) {
                    q[col] = row;
                    visit(q, col + 1, solutions);
                }
            }
        }
        let mut solutions = Vec::new();
        visit(&mut vec![0; n], 0, &mut solutions);
        solutions
    }

    fn canonical_form(q: &[usize]) -> Vec<usize> {
        let m = q.len() - 1;
        let mut best = q.to_vec();
        // Quatre rotations, chacune avec ou sans miroir gauche-droite.
        for turns in 0..4 {
            for mirror in [false, true] {
                let mut image = vec![0; q.len()];
                for (col, &row) in q.iter().enumerate() {
                    let (mut x, mut y) = (col, row);
                    for _ in 0..turns { (x, y) = (m - y, x); }
                    if mirror { x = m - x; }
                    image[x] = y;
                }
                if image < best { best = image; }
            }
        }
        best
    }

    let n = 8;
    // Enumeration via CP-SAT dans le jumeau Python ; ici, enumeration par backtracking.
    let start = Instant::now();
    let solutions = enumerate_nqueens_solutions_backtracking(n);
    let ms = start.elapsed().as_secs_f64() * 1_000.0;
    println!("Solutions totales : {} (temps {ms:.3} ms)", solutions.len());

    // Comptage par contraintes de brisure simples
    let simple: Vec<_> = solutions.iter().filter(|q| q[0] < q[n - 1] && q[0] < n / 2).collect();
    println!("Avec contraintes simples : {}", simple.len());

    // Solutions fondamentales (reduction par symetries)
    let classes: BTreeSet<_> = solutions.iter().map(|q| canonical_form(q)).collect();
    let simple_classes: BTreeSet<_> = simple.iter().map(|q| canonical_form(q)).collect();
    println!("Solutions fondamentales : {}", classes.len());
    assert_eq!((solutions.len(), simple.len(), classes.len()), (92, 35, 12));
    assert_eq!(simple_classes, classes, "La brisure simple doit couvrir chaque classe");
    println!("OK : on trouve bien 12 solutions fondamentales");
};


Solutions totales : 92 (temps 0.447 ms)


Avec contraintes simples : 35


Solutions fondamentales : 12


OK : on trouve bien 12 solutions fondamentales


### Exemple guide 3b : brisure de symetrie directe dans CP-SAT

*Contribution d'un groupe d'étudiants (PR #20937), intégrée comme exemple guidé.*

**Enonce** : au lieu d'enumerer toutes les solutions puis de reduire
post-facto, ajoutez les contraintes de symetrie **directement dans le
modèle CP-SAT** pour n'enumerer que les solutions fondamentales.

**Consignes** :
1. Ajoutez les contraintes $q_0 < q_{N-1}$ et $q_0 < N/2$ au modèle et énumérez :
   le compte donne **35** solutions pour $N = 8$, et non 12. Ces deux contraintes
   sont valides (aucune classe de symétrie n'est perdue) mais incomplètes :
   plusieurs représentants par classe survivent.
2. Complétez la brisure avec un lex-leader sur les 7 symétries non triviales du
   carré (groupe D4) : l'énumération retombe exactement aux **12** solutions
   fondamentales de l'exemple guide 3.
3. Mesurez le gain de temps par rapport a l'énumération complète.

In [28]:
// EXEMPLE GUIDE 3b : brisure de symetrie dans CP-SAT (contribution etudiante, PR #20937)
{
    use cp_sat::builder::{CpModelBuilder, LinearExpr};
    use cp_sat::proto::{CpSolverStatus, SatParameters};
    use std::collections::BTreeSet;
    use std::time::Instant;

    type Solutions = BTreeSet<Vec<usize>>;

    #[derive(Clone, Copy, Debug)]
    enum Mode { SansBrisure, Simple, Lex }

    fn scaled(expr: &LinearExpr, coefficient: usize) -> LinearExpr {
        // Construire un multiple entier avec les opérations linéaires du builder.
        (0..coefficient).fold(LinearExpr::from(0i64), |sum, _| sum + expr.clone())
    }

    fn add_lex_leq(model: &mut CpModelBuilder, x: &[LinearExpr], y: &[LinearExpr], m: usize) {
        // x[i] et y[i] sont dans 0..=m, donc leur différence est dans -m..=m.
        // Une borne serrée permet d'activer les inégalités sans only_enforce_if.
        let mut prefix: Option<LinearExpr> = None;
        for (i, (a, b)) in x.iter().zip(y).enumerate() {
            let d = a.clone() + -b.clone();
            if let Some(p) = &prefix {
                // p = 1 => d <= 0 ; p = 0 => d <= m, déjà vrai par les domaines.
                model.add_le(d.clone() + scaled(p, m), m as i64);
            } else {
                // Le préfixe vide est toujours égal.
                model.add_le(d.clone(), 0i64);
            }
            if i + 1 == x.len() { break; }

            // Un seul des trois cas d < 0, d = 0 et d > 0 est possible.
            // Les trois booléennes sont donc fixées par d, sans doublons à l'énumération.
            let lt = LinearExpr::from(model.new_bool_var());
            let e = LinearExpr::from(model.new_bool_var());
            let gt = LinearExpr::from(model.new_bool_var());
            model.add_eq(lt.clone() + e.clone() + gt.clone(), 1i64);
            model.add_le(d.clone() + scaled(&lt, m + 1), m as i64);
            model.add_le(-d.clone() + scaled(&gt, m + 1), m as i64);
            model.add_le(d.clone() + scaled(&e, m), m as i64);
            model.add_le(-d + scaled(&e, m), m as i64);

            prefix = if let Some(p) = prefix {
                let p_next = LinearExpr::from(model.new_bool_var());
                // p_next => p et e
                model.add_le(p_next.clone(), p.clone());
                model.add_le(p_next.clone(), e.clone());
                // p et e => p_next
                model.add_le(p + e, p_next.clone() + 1i64);
                Some(p_next)
            } else {
                Some(e)
            };
        }
    }

    fn enumerate_nqueens_symmetry(n: usize, mode: Mode) -> (Solutions, f64, CpSolverStatus) {
        assert!((8..=12).contains(&n));
        let mut model = CpModelBuilder::default();
        let m = n - 1;
        let q: Vec<_> = (0..n).map(|_| model.new_int_var(std::iter::once((0i64, m as i64)))).collect();
        model.add_all_different(q.iter().copied());
        model.add_all_different(q.iter().copied().enumerate().map(|(i, v)| LinearExpr::from(v) + i as i64));
        model.add_all_different(q.iter().copied().enumerate().map(|(i, v)| LinearExpr::from(v) + -(i as i64)));

        match mode {
            Mode::SansBrisure => {}
            Mode::Simple => {
                model.add_le(LinearExpr::from(q[0]) + 1i64, q[m]);
                model.add_le(q[0], (n / 2 - 1) as i64);
            }
            Mode::Lex => {
                // inv[r] = colonne de la reine situee sur la ligne r
                let inv: Vec<_> = (0..n).map(|_| model.new_int_var(std::iter::once((0i64, m as i64)))).collect();
                // La même matrice booléenne code q par colonne et inv par ligne.
                // Une seule case active par colonne et par ligne fixe tous les auxiliaires.
                let cell: Vec<Vec<LinearExpr>> = (0..n).map(|_| {
                    (0..n).map(|_| LinearExpr::from(model.new_bool_var())).collect()
                }).collect();
                for c in 0..n {
                    let one = cell[c].iter().cloned().reduce(|a, b| a + b).unwrap();
                    model.add_eq(one, 1i64);
                    let row = (0..n).map(|r| scaled(&cell[c][r], r)).reduce(|a, b| a + b).unwrap();
                    model.add_eq(q[c], row);
                }
                for r in 0..n {
                    let one = (0..n).map(|c| cell[c][r].clone()).reduce(|a, b| a + b).unwrap();
                    model.add_eq(one, 1i64);
                    let col = (0..n).map(|c| scaled(&cell[c][r], c)).reduce(|a, b| a + b).unwrap();
                    model.add_eq(inv[r], col);
                }
                let x: Vec<_> = q.iter().copied().map(LinearExpr::from).collect();
                let inverse: Vec<_> = inv.iter().copied().map(LinearExpr::from).collect();
                let complement = |v: &LinearExpr| -v.clone() + m as i64;
                // Image de la solution par chaque symetrie, ecrite colonne par colonne
                let images: [(&str, Vec<LinearExpr>); 7] = [
                    ("miroir_vertical", (0..n).map(|c| x[m - c].clone()).collect()), // x -> m - x
                    ("miroir_horizontal", (0..n).map(|c| complement(&x[c])).collect()), // y -> m - y
                    ("rotation_180", (0..n).map(|c| complement(&x[m - c])).collect()),
                    ("transposee", (0..n).map(|c| inverse[c].clone()).collect()), // (x, y) -> (y, x)
                    ("rotation_90", (0..n).map(|c| inverse[m - c].clone()).collect()), // (x, y) -> (m - y, x)
                    ("rotation_270", (0..n).map(|c| complement(&inverse[c])).collect()), // (x, y) -> (y, m - x)
                    ("anti_transposee", (0..n).map(|c| complement(&inverse[m - c])).collect()),
                ];
                // Lex-leader : on ne garde que le plus petit element de chaque orbite
                for (_, image) in &images { add_lex_leq(&mut model, &x, image, m); }
            }
        }

        let params = SatParameters {
            max_time_in_seconds: Some(60.0),
            num_search_workers: Some(1),
            enumerate_all_solutions: Some(true),
            fill_additional_solutions_in_response: Some(true),
            solution_pool_size: Some(32_768),
            ..Default::default()
        };
        let start = Instant::now();
        let response = model.solve_with_parameters(&params);
        let ms = start.elapsed().as_secs_f64() * 1_000.0;
        let mut solutions = Solutions::new();
        for values in response.additional_solutions.iter().map(|s| s.values.as_slice())
            .chain(std::iter::once(response.solution.as_slice()))
        {
            if let Some(rows) = values.get(..n) {
                solutions.insert(rows.iter().map(|&r| r as usize).collect());
            }
        }
        (solutions, ms, response.status())
    }

    fn canonical_form(q: &[usize]) -> Vec<usize> {
        let m = q.len() - 1;
        let mut best = q.to_vec();
        // Calcul indépendant sur les coordonnées, après la résolution.
        for turns in 0..4 {
            for mirror in [false, true] {
                let mut image = vec![0; q.len()];
                for (col, &row) in q.iter().enumerate() {
                    let (mut x, mut y) = (col, row);
                    for _ in 0..turns { (x, y) = (m - y, x); }
                    if mirror { x = m - x; }
                    image[x] = y;
                }
                if image < best { best = image; }
            }
        }
        best
    }

    fn best_time_ms(n: usize, mode: Mode) -> (Solutions, f64) {
        // Trois répétitions jusqu'à N=10, puis une pour les énumérations longues.
        let repeats = if n <= 10 { 3 } else { 1 };
        let mut best: Option<(Solutions, f64)> = None;
        for _ in 0..repeats {
            let (solutions, ms, status) = enumerate_nqueens_symmetry(n, mode);
            assert_eq!(status, CpSolverStatus::Optimal, "Énumération incomplète pour N={n}, mode {mode:?}");
            if best.as_ref().map_or(true, |(_, previous_ms)| ms < *previous_ms) {
                best = Some((solutions, ms));
            }
        }
        best.expect("Au moins une mesure")
    }

    // --- Etapes 1 et 2 : N = 8 ---
    let (all8, ta8) = best_time_ms(8, Mode::SansBrisure);
    let (simple8, ts8) = best_time_ms(8, Mode::Simple);
    let (lex8, tl8) = best_time_ms(8, Mode::Lex);
    let classes_all: Solutions = all8.iter().map(|q| canonical_form(q)).collect();
    let classes_simple: Solutions = simple8.iter().map(|q| canonical_form(q)).collect();
    println!("N = 8");
    println!("  Sans brisure            : {} solutions (OPTIMAL)", all8.len());
    println!("  Contraintes de l'enonce : {} solutions (OPTIMAL), couvrant {}/{} classes",
        simple8.len(), classes_simple.len(), classes_all.len());
    println!("  Lex-leader D4           : {} solutions (OPTIMAL)", lex8.len());
    // Verifications croisees avec l'exemple guide (reduction post-facto)
    assert_eq!(simple8.len(), 35); // 35 dans l'exemple
    assert_eq!(classes_simple, classes_all); // aucune classe perdue
    assert_eq!(lex8.len(), 12); // 12 fondamentales
    assert_eq!(lex8, classes_all); // exactement les formes canoniques
    println!("  Les 12 solutions lex-leader sont exactement les formes canoniques de l'exemple");

    // --- Etape 3 : gain de temps, sur plusieurs tailles ---
    // Valeurs de reference (OEIS A000170 et A002562)
    let cases = [(8, 92, 35, 12), (9, 352, 131, 46), (10, 724, 289, 92),
        (11, 2_680, 958, 341), (12, 14_200, 5_564, 1_787)];
    let counts8 = (all8.len(), simple8.len(), lex8.len());
    println!("\n  N |   total      ms |  simple      ms |     lex      ms | gain simple | gain lex");
    for (n, total, simple, lex) in cases {
        // Réutiliser les mesures de N=8 faites dans cette cellule.
        let (counts, ta, ts, tl) = if n == 8 {
            (counts8, ta8, ts8, tl8)
        } else {
            let (all_solutions, ta) = best_time_ms(n, Mode::SansBrisure);
            let (simple_solutions, ts) = best_time_ms(n, Mode::Simple);
            let (lex_solutions, tl) = best_time_ms(n, Mode::Lex);
            let classes_all: Solutions = all_solutions.iter().map(|q| canonical_form(q)).collect();
            let classes_simple: Solutions = simple_solutions.iter().map(|q| canonical_form(q)).collect();
            assert_eq!(classes_simple, classes_all, "Couverture des classes pour N={n}");
            assert_eq!(lex_solutions, classes_all, "Formes canoniques pour N={n}");
            ((all_solutions.len(), simple_solutions.len(), lex_solutions.len()), ta, ts, tl)
        };
        assert_eq!(counts, (total, simple, lex), "Comptes pour N={n}");
        let (a, s, l) = counts;
        println!("{n:>3} | {a:>7} {ta:>7.1} | {s:>7} {ts:>7.1} | {l:>7} {tl:>7.1} | {:>10.1}x | {:>7.1}x", ta / ts, ta / tl);
    }
    println!("Comptes conformes aux valeurs de reference (OEIS A000170 / A002562)");
};


N = 8


  Sans brisure            : 92 solutions (OPTIMAL)


  Contraintes de l'enonce : 35 solutions (OPTIMAL), couvrant 12/12 classes


  Lex-leader D4           : 12 solutions (OPTIMAL)


  Les 12 solutions lex-leader sont exactement les formes canoniques de l'exemple


  N |   total      ms |  simple      ms |     lex      ms | gain simple | gain lex


... (2 outputs left out)

 10 |     724   741.0 |     289   801.8 |      92   423.0 |        0.9x |     1.8x


 11 |    2680  4088.4 |     958  1575.0 |     341  1092.7 |        2.6x |     3.7x


 12 |   14200 54216.3 |    5564 11909.0 |    1787  7282.2 |        4.6x |     7.4x


Comptes conformes aux valeurs de reference (OEIS A000170 / A002562)


### Interprétation : exemple guide 3b (brisure de symétrie dans CP-SAT)

**Sortie obtenue** : avec les deux contraintes de la consigne ($q_0 < q_{N-1}$ et $q_0 < N/2$), CP-SAT énumère **35 solutions, et non 12** : c'est le même 35 que la ligne « contraintes simples » de l'exemple guidé. Le compte ne correspond donc **pas** aux solutions fondamentales. Avec un lex-leader complet, CP-SAT énumère exactement **12** solutions, qui sont précisément les formes canoniques de l'exemple.

| Brisure | Solutions ($N=8$) | Valide (aucune classe perdue) | Complète (une solution par classe) |
|---|---|---|---|
| Aucune | 92 | oui | non |
| Consigne : $q_0 < q_{N-1}$, $q_0 < N/2$ | 35 | oui (12/12 classes représentées) | non |
| Lex-leader sur le groupe D4 | 12 | oui | **oui** |

**Pourquoi 35 ?** Le carré a 8 symétries (groupe diédral D4 : 4 rotations, 4 réflexions). Les 92 solutions forment 12 classes : 11 de 8 solutions et une de 4, invariante par rotation de 180°. La contrainte $q_0 < q_{N-1}$ casse le miroir gauche-droite et $q_0 < N/2$ casse en partie le miroir haut-bas. Les rotations de 90° et les deux transpositions, qui échangent lignes et colonnes, ne sont pas cassées : il reste plusieurs représentants par classe.

**Brisure complète (lex-leader, Crawford, Ginsberg, Luks & Roy, 1996)** : pour chacune des 7 symétries non triviales $g$, on impose $q \leq_{lex} g(q)$. Seul le plus petit élément de chaque classe, au sens lexicographique, survit. Les symétries qui échangent lignes et colonnes s'expriment avec la permutation inverse (`add_inverse`). Les booléennes auxiliaires de l'ordre lexicographique sont définies par des équivalences : sinon, une même solution pourrait être comptée plusieurs fois.

**Validation** : pour $N = 8$ à $12$, les comptes obtenus (92 / 352 / 724 / 2 680 / 14 200 au total ; 12 / 46 / 92 / 341 / 1 787 avec lex-leader) correspondent aux valeurs de référence OEIS A000170 et A002562. Les contraintes de la consigne restent valides pour toutes ces tailles.

**Gain de temps** (*machine-dep*) : pour $N \geq 9$, le lex-leader accélère l'énumération d'un facteur compris entre 1,8 et 7,4 selon la taille dans l'exécution committée (colonne « gain lex » du tableau ci-dessus). Face à la réduction du nombre de solutions (facteur 7,7 à 8 selon $N$), ce gain reste nettement inférieur pour $N \leq 10$ (1,8 à 3,0) et ne s'en approche qu'à $N = 12$ (7,4) : le solveur explore encore des branches qu'il rejette, et doit propager les 7 contraintes lexicographiques. Les contraintes de la consigne donnent un gain plus faible et irrégulier, qui varie selon $N$ d'environ 1 à 4,6 fois (colonne « gain simple »). À $N = 8$, tous les temps sont surtout dominés par le coût fixe de construction et de lancement du solveur.

> **À retenir** : une brisure de symétrie se vérifie par le **compte** de solutions. Une brisure valide et complète doit donner exactement le nombre de classes (12 pour $N = 8$). Les deux contraintes proposées sont valides mais incomplètes. Seule une brisure portant sur tout le groupe de symétrie isole les solutions fondamentales.

### Exercice 3c : quelles symétries rapportent le plus ?

**Enonce** : l'exemple guide 3b impose les 7 contraintes lexicographiques d'un coup.
Chacune coûte à la propagation. Mesurez ce que chaque symétrie apporte **seule**,
puis cherchez le plus petit sous-ensemble qui suffit.

**Consignes** :
1. Pour chacune des 7 symétries de l'exemple guide 3b (miroir vertical, miroir
   horizontal, rotation 180°, transposée, rotation 90°, rotation 270°,
   anti-transposée), énumérez les solutions de $N = 8$ avec la seule contrainte
   $q \leq_{lex} g(q)$ : combien de solutions survivent ?
2. Parcourez les sous-ensembles de symétries ($2^7 - 1$ possibilités) et identifiez
   les plus petits dont les contraintes conjointes laissent **exactement 12**
   solutions. Les deux générateurs de D4 (par exemple rotation 90° et un miroir)
   suffisent-ils ?
3. Pour 2 ou 3 sous-ensembles (y compris les 7 contraintes), comparez le temps
   d'énumération à $N = 9$ et $N = 10$ : plus de contraintes signifie-t-il
   toujours plus vite ?

> **Indice** : `enumerate_nqueens_symmetry` de l'exemple guide 3b construit le
> tableau `images` des 7 symétries — parametrez-le par sous-ensemble.

In [29]:
// Exercice 3c : quelles symetries rapportent le plus ?
// TODO étudiant : parametriser enumerate_nqueens_symmetry par sous-ensemble.
{
    use cp_sat::proto::CpSolverStatus;

    type Enumeration = (Vec<Vec<usize>>, f64, CpSolverStatus);

    fn enumerate_nqueens_subset(_n: usize, _sym_names: &[&str], _time_limit_s: f64) -> Option<Enumeration> {
        // TODO étudiant : modele N-Reines + lex par symetrie choisie.
        None
    }

    // --- Etape 1 : chaque symetrie seule, N = 8 ---
    // TODO étudiant : afficher, par symetrie, le nombre de solutions qui survivent.

    // --- Etape 2 : plus petits sous-ensembles laissant exactement 12 solutions ---
    // TODO étudiant : parcourir les 2^7 - 1 sous-ensembles avec des masques de bits.

    // --- Etape 3 : temps d'enumeration a N = 9, 10 pour quelques sous-ensembles ---
    // TODO étudiant : comparer temps et nombre de contraintes ; compter aussi les
    // solutions fondamentales attendues (OEIS A002562 : 46 pour N = 9, 92 pour N = 10).
    let _ = enumerate_nqueens_subset(8, &[], 30.0);
    println!("Exercice a completer");
};


Exercice a completer


***

## Conclusion et recapitulatif

### Tableau de synthese des approches

| Critere | Backtracking | Min-Conflicts | CP-SAT |
|---------|-------------|---------------|--------|
| Completude | Oui | Non | Oui |
| Complexite N-Reines | Exponentielle | Quasi-lineaire | Variable |
| Enumeration | Possible (lent) | Non | Oui (natif) |
| Difficulte d'implementation | Moyenne | Facile | Facile (declaratif) |
| Scalabilite | Faible (N < 30) | Excellente (N > 10^6) | Bonne (N > 10^3) |
| Pedagogie | Comprendre les CSP | Comprendre la recherche locale | Utiliser un outil pro |
### References

- Russell, S. & Norvig, P. *Artificial Intelligence: A Modern Approach*, 4th ed., Chapitre 6
- Minton, S., Johnston, M.D., Philips, A.B. & Laird, P. (1992). *Min-Conflicts: A Simple, Complète, Backtrack-Free Search Procedure*. Artificial Intelligence, 58(1-3)
- [OR-Tools CP-SAT Documentation](https://developers.google.com/optimization/cp/cp_solver)

## À retenir

- **L'heuristique change la taille de l'arbre, pas la solution.** À N = 8, le backtracking simple explore **876** nœuds, MRV **75** et FC+MRV **75** ; à N = 12, **3066** contre **153** ; à N = 16, **160712** contre **44** (FC+MRV **39**). La solution reste la même : [0, 4, 7, 5, 2, 6, 1, 3].
- **Min-Conflicts tient l'échelle là où le backtracking décroche.** L'instance N = 500 est résolue avec **346** itérations.
- **CP-SAT prouve, mais ne passe pas à l'échelle.** N = 8 : statut **OPTIMAL** et énumération des **92** solutions ; à N = 500, le solveur atteint la limite de temps et rend **UNKNOWN**.
- **Le seuil de croisement backtracking / min-conflicts n'est pas stable.** Dans le jumeau Python, aucun croisement jusqu'à N = 50, et le backtracking expire sur **4/8** tailles au-delà (limite 5 s). Dans ce jumeau Rust, le basculement se produit dès N = 45–50 (écart de 0,138 ms à N = 45, puis de 1,739 ms à N = 50, machine-dépendant) et le backtracking n'expire que sur **2/8** tailles (110 et 120) ; min-conflicts résout toutes les tailles testées des deux côtés.
- **La brisure de symétrie est un multiplicateur.** Sans brisure : **92** solutions ; avec les seules contraintes de l'énoncé : **35** ; avec un lex-leader complet sur le groupe D4 : **12**, qui sont exactement les formes canoniques.

## Vérifiez votre compréhension

1. À N = 8, pourquoi le backtracking simple explore-t-il **876** nœuds quand MRV n'en explore que **75** ?

   <details><summary>Réponse</summary>

   MRV (*Minimum Remaining Values*) choisit à chaque pas la variable la plus contrainte : l'échec est détecté tôt, l'arbre est élagué avant de s'étendre. Forward Checking ajoute la propagation des contraintes et donne le même compte sur cette instance (**75**), mais l'ordre de Branch n'est pas le même : à N = 16, les deux se séparent (**44** contre **39**).

   </details>

2. La cellule « Performance CP-SAT » annonce un temps pour N = 8 alors que le tableau comparatif en affiche un autre : lequel fait foi ?

   <details><summary>Réponse</summary>

   Les deux, dans deux cadres différents. L'un est la durée de résolution rapportée par la cellule CP-SAT (statut **OPTIMAL**), l'autre est le temps mesuré dans le banc comparatif, qui lance les trois approches sur les mêmes tailles. La règle du carnet : la sortie d'une cellule est la mesure, un tableau de banc est une comparaison — on ne les confond pas.

   </details>

3. Si le tableau comparatif montre min-conflicts plus rapide que CP-SAT à N = 100, peut-on en conclure que min-conflicts est le meilleur solveur ?

   <details><summary>Réponse</summary>

   Non : les deux répondent à des questions différentes. Min-Conflicts rend **une** solution valide sans prouver quoi que ce soit ; CP-SAT rend une solution **OPTIMAL** et sait énumérer les **92** solutions. Et l'échelle ne les départage pas dans le même sens : à N = 500, CP-SAT rend **UNKNOWN** dans le jumeau Python. Le choix se prend sur le besoin — une solution, ou une preuve.

   </details>

4. Le lex-leader trouve **12** solutions là où les contraintes de l'énoncé en trouvaient **35** : pourquoi l'énoncé ne suffit-il pas ?

   <details><summary>Réponse</summary>

   Les **92** solutions se répartissent en **12** classes de symétrie du groupe D4 (ordre **8**). Les contraintes de l'énoncé ne cassent que le miroir gauche-droite et, partiellement, le miroir haut-bas : les rotations de 90° et les deux transpositions subsistent, donc plusieurs représentants par classe survivent (**35**). Le lex-leader impose les **7** symétries non triviales et ne laisse qu'un représentant par classe.

   </details>

## Pour aller plus loin

**Carnet suivant.** [App-2 : Coloration de graphes](App-02-GraphColoring-Python.ipynb) -- un autre problème CSP classique avec applications cartographiques

**Lectures.**

- Les références complètes du carnet sont dans la section `### References` de la conclusion : Russell & Norvig, *Artificial Intelligence: A Modern Approach*, chapitre 6, pour la formalisation CSP, et Minton et al. (1992) pour l'algorithme Min-Conflicts.
- La documentation CP-SAT d'OR-Tools accompagne la lecture du solveur : c'est le même moteur que le twin C# du carnet, qui l'invoque via Google.OrTools, et que le [jumeau Python](App-01-NQueens-Python.ipynb), qui l'invoque via `ortools.sat.python.cp_model`.

**Carnets voisins.**

- [App-2 : Graph Coloring](App-02-GraphColoring-Python.ipynb) -- un autre problème CSP classique avec applications cartographiques
- [CSP-2 : Consistency](../Part2-CSP/CSP-2-Consistency.ipynb) -- propagation de contraintes (AC-3, MAC)
- [CSP-3 : Advanced](../Part2-CSP/CSP-3-Advanced.ipynb) -- contraintes globales, Large Neighborhood Search
- [App-1-NQueens-CSharp](App-01-NQueens-CSharp.ipynb) -- le jumeau C#, qui déroule les mêmes algorithmes à la main avant de brancher CP-SAT.

***

**Navigation** : [Index](../README.md) | [Part2-CSP](../Part2-CSP/README.md) | [App-2 GraphColoring >>](App-02-GraphColoring-Python.ipynb)

> Carnet issu de l'EPIC #27132. [Spécification #28086](https://github.com/jsboige/CoursIA/issues/28086) · [Gabarit de portage Rust](../../../docs/reference/rust-porting-spec.md).